# Rocket Control

## Contents
1. [Rocket Sprite](#rocket-sprite)
2. [Simplified 1D Case](#simplified-1d-case-constant-mass-full-thrust-no-animation-only-calculation)
3. [The 1D, Time-Dependent Mass Case](#the-1d-time-dependent-mass-case)
4. [Attitude Control System (Rotational Dynamics)](#attitude-control-system-rotation)
5. [Full 2D Case](#full-2d-case) \
5.1 [SLSQP Implementation](#slsqp-implementation) \
5.2 [CasADi Implementation](#casadi-implementation)
6. [Test](#testing-convergence--reachable-set)

## Rocket Sprite

Run this cell, specify the angle of the rocket and the gimbal, and you will see a rocket sprite!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.colors import LinearSegmentedColormap

def rotate_point(x, y, pivot_x, pivot_y, angle_deg):
    """Rotates a point around a pivot by a given angle in degrees."""
    angle_rad = np.radians(angle_deg)
    nx, ny = x - pivot_x, y - pivot_y
    s, c = np.sin(-angle_rad), np.cos(-angle_rad)
    rx = nx * c - ny * s
    ry = nx * s + ny * c
    return rx + pivot_x, ry + pivot_y

def get_rotated_points(points_list, pivot, angle):
    return [rotate_point(p[0], p[1], pivot[0], pivot[1], angle) for p in points_list]

# Setup 
pivot = (0, 0)
rocket_height = 70
rocket_width = rocket_height / 10
nose_height = 15
engine_height = 10
engine_width_bottom = rocket_width * 1.2
engine_width_top = rocket_width * 0.65

# Flame variables
flame_height = 20  # Adjust for "throttle"
flame_width = engine_width_bottom * 0.6

# Inputs
body_tilt = float(input("Enter rocket tilt angle (deg): "))
gimbal_angle = float(input("Enter engine gimbal angle (max +/-10 deg): "))
gimbal_angle = max(-10, min(10, gimbal_angle))

fig, ax = plt.subplots(figsize=(8, 8))
fig_size = rocket_height + 30
ax.set_xlim(-fig_size, fig_size)
ax.set_ylim(-fig_size, fig_size)
ax.set_aspect('equal')
ax.axis('off')

# Gradient Background
custom_cmap = LinearSegmentedColormap.from_list("sky", ["#102C57", "#000000"])
gradient = np.linspace(0, 1, 100).reshape(-1, 1)
ax.imshow(gradient, extent=[-fig_size, fig_size, -fig_size, fig_size], origin='lower', cmap=custom_cmap, aspect='auto')

# 1. Define Raw Coordinates 
body_raw = [[-rocket_width, -rocket_height/2], [rocket_width, -rocket_height/2], 
            [rocket_width, rocket_height/2], [-rocket_width, rocket_height/2]]
nose_raw = [[-rocket_width, rocket_height/2], [rocket_width, rocket_height/2], 
            [0, rocket_height/2 + nose_height]]

# Engine attachment point (on the body)
engine_attach_point = (0, -rocket_height/2 + engine_height*0.1)
skirt_raw = [[-engine_width_bottom, -rocket_height/2 - engine_height], 
             [engine_width_bottom, -rocket_height/2 - engine_height], 
             [engine_width_top, -rocket_height/2 + engine_height*0.1], 
             [-engine_width_top, -rocket_height/2 + engine_height*0.1]]

# Flame Raw (Centered at the base of the engine)
flame_base_y = -rocket_height/2 - engine_height
flame_raw = [[-flame_width, flame_base_y], 
             [flame_width, flame_base_y], 
             [0, flame_base_y - flame_height]]

# 2. Process Rotations (Nested) 
# A. Rotate engine AND flame around the engine attachment point
skirt_gimbaled = get_rotated_points(skirt_raw, engine_attach_point, gimbal_angle)
flame_gimbaled = get_rotated_points(flame_raw, engine_attach_point, gimbal_angle)

# B. Rotate everything around the main body pivot
body_rot = get_rotated_points(body_raw, pivot, body_tilt)
nose_rot = get_rotated_points(nose_raw, pivot, body_tilt)
skirt_final = get_rotated_points(skirt_gimbaled, pivot, body_tilt)
flame_final = get_rotated_points(flame_gimbaled, pivot, body_tilt)

# 3. Draw 
# Flame drawn first so it sits "behind" or flush with the engine
ax.add_patch(Polygon(flame_final, fc='orange', ec='red', lw=1.5, zorder=9))
ax.add_patch(Polygon(skirt_final, fc="#D0D0D0", ec='black', lw=1.2, zorder=10))
ax.add_patch(Polygon(body_rot, fc="#FFFFFF", ec='black', lw=1.2, zorder=11))
ax.add_patch(Polygon(nose_rot, fc="#FFFFFF", ec='black', lw=1.2, zorder=12))

plt.title(f"Body Tilt: {body_tilt}° | Gimbal: {gimbal_angle}°", color='white', fontsize=12)
plt.show()

## Simplified 1D Case: Constant Mass, Full Thrust (No animation, only calculation)
First, let's consider the 1D system of a falling rocket, which can be controlled via a thruster (with a maximum thrust). An initial fuel amount is given. As a simplification, the mass is held constant, which doesn't take into account the continuously burned fuel.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# User inputs
h0 = float(input("Initial height (m): "))
initial_velocity_down = float(input("Initial downward velocity (m/s): "))
g = float(input("Gravity (m/s^2): "))
dry_mass = float(input("Dry mass (kg): "))
fuel_mass = float(input("Fuel mass (kg): "))
u_max = float(input("Maximum thrust acceleration (m/s^2): "))
v_e = float(input("Exhaust velocity (m/s): "))

# print initial condition summary
print("="*70)
print("INITIAL CONDITIONS")
print("="*70)
print(f"Initial height: {h0} m \nInitial velocity: {initial_velocity_down} m/s \n" + 
      f"Gravity: {g} m/s^2 \nDry Mass: {dry_mass} kg \nFuel Mass: {fuel_mass} kg \n" +
      f"Total Mass: {dry_mass + fuel_mass} kg \nMaximum Thrust Acceleration: {u_max} m/s^2 \n" + 
      f"Exhaust Velocity: {v_e} m/s")
print("="*70)
print("RESULTS")
print("="*70)

# Derived
m0 = dry_mass + fuel_mass
fuel_amount = v_e * math.log(m0 / dry_mass) if fuel_mass > 0 else 0  # Available delta-v

# Set conventions: positive upwards
v0 = -initial_velocity_down  # v0 < 0 for downward

if u_max <= g:
    print("Cannot land: maximum thrust acceleration must at least exceed gravity.")
else:
    a_net = u_max - g
    h_req_initial = 0.5 * (-v0)**2 / a_net
    if h0 < h_req_initial:
        print("Cannot land: initial height too low for the velocity.")
    else:
        # Solve quadratic for coast time
        A = -g * u_max
        B = 2 * v0 * u_max
        C = 2 * a_net * h0 - v0**2
        D = B**2 - 4 * A * C
        if D < 0:
            print("No real solution (unexpected).")
        else:
            sqrt_D = math.sqrt(D)
            t1 = (-B + sqrt_D) / (2 * A)
            t2 = (-B - sqrt_D) / (2 * A)
            candidates = [t for t in [t1, t2] if t >= 0]
            if not candidates:
                print("No positive coast time solution.")
            else:
                t_coast = min(candidates)  # The earliest valid time to start burn
                v_b = v0 - g * t_coast
                y_b = h0 + v0 * t_coast - 0.5 * g * t_coast**2
                t_burn = -v_b / a_net
                fuel_used = u_max * t_burn
                if fuel_used > fuel_amount:
                    print(f"Not enough fuel: requires {fuel_used:.2f} m/s delta-v, but only {fuel_amount:.2f} available.")
                else:
                    # Approximate fuel mass used (for info, since constant mass model)
                    fuel_used_kg_approx = fuel_mass * (1 - math.exp(-fuel_used / v_e))
                    print(f"Can land optimally using approx {fuel_used_kg_approx:.2f} kg of fuel ({fuel_used:.2f} m/s delta-v).")
                    print(f"Coast for {t_coast:.2f} seconds, then full thrust for {t_burn:.2f} seconds.")

                    # Simulate and plot the trajectory
                    t_total = t_coast + t_burn
                    times = np.linspace(0, t_total, 500)
                    heights = []
                    velocities = []
                    thrusts = []
                    for t in times:
                        if t <= t_coast:
                            thrust = 0
                            height = h0 + v0 * t - 0.5 * g * t**2
                            velocity = v0 - g * t
                        else:
                            t_rel = t - t_coast
                            thrust = u_max
                            height = y_b + v_b * t_rel + 0.5 * (u_max - g) * t_rel**2
                            velocity = v_b + (u_max - g) * t_rel
                        heights.append(height)
                        velocities.append(velocity)
                        thrusts.append(thrust)

                    # Verify landing
                    if abs(heights[-1]) < 1e-6 and abs(velocities[-1]) < 1e-6:
                        print("Simulation confirms soft landing.")
                    else:
                        print("Simulation error in landing conditions.")

                    # Plot
                    fig, axs = plt.subplots(3, 1, figsize=(8, 10))
                    axs[0].axvline(x=t_coast, color="lightcoral", linestyle="--", label="Thrust activation")
                    axs[0].plot(times, heights)
                    axs[0].set_ylabel('Height (m)')
                    axs[0].grid(True)
                    axs[0].legend()
                    axs[1].axvline(x=t_coast, color="lightcoral", linestyle="--", label="Thrust activation")
                    axs[1].plot(times, velocities)
                    axs[1].set_ylabel('Velocity (m/s)')
                    axs[1].grid(True)
                    axs[1].legend()
                    axs[2].axvline(x=t_coast, color="lightcoral", linestyle="--", label="Thrust activation")
                    axs[2].plot(times, thrusts)
                    axs[2].set_ylabel('Thrust accel (m/s^2)')
                    axs[2].set_xlabel('Time (s)')
                    axs[2].grid(True)
                    axs[2].legend()
                    plt.tight_layout()
                    plt.show()

## The 1D, Time-Dependent Mass Case
Let's now consider how the mass changes along the trajectory as the rocket burns fuel, using the Tsiolkovsky rocket equation:

$$\dot{m} = -\frac{T}{v_e}$$

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Rectangle, Polygon, FancyArrowPatch
from matplotlib.colors import LinearSegmentedColormap
from scipy.interpolate import interp1d

# User inputs
h0 = float(input("Initial height (m): "))
initial_velocity_down = float(input("Initial downward velocity (m/s): "))
g = float(input("Gravity (m/s^2): "))
dry_mass = float(input("Dry mass (kg): "))
fuel_mass = float(input("Fuel mass (kg): "))
u_max = float(input("Maximum initial thrust acceleration (m/s^2): "))
v_e = float(input("Exhaust velocity (m/s): "))

# print initial condition summary
print("="*70)
print("INITIAL CONDITIONS")
print("="*70)
print(f"Initial height: {h0} m \nInitial velocity: {initial_velocity_down} m/s \n" +
      f"Gravity: {g} m/s^2 \nDry Mass: {dry_mass} kg \nFuel Mass: {fuel_mass} kg \n" +
      f"Total Mass: {dry_mass + fuel_mass} kg \nMaximum Thrust Acceleration: {u_max} m/s^2 \n" +
      f"Exhaust Velocity: {v_e} m/s")
print("="*70)
print("RESULTS")
print("="*70)

# Derived
m0 = dry_mass + fuel_mass
m_dry = dry_mass
T = u_max * m0  # constant thrust force (N)

# Set conventions: positive upwards
v0 = -initial_velocity_down  # v0 < 0 for downward

if u_max <= g:
    print("Cannot land: maximum initial thrust acceleration must exceed gravity.")
else:
    # Time to impact without thrust
    v_down = initial_velocity_down
    t_max = (v_down + math.sqrt(v_down**2 + 2 * g * h0)) / g

    def get_projected_min_h(h_b, v_b, m_b, T, v_e, g, dt=0.001):
        h = h_b
        v = v_b
        m = m_b
        t = 0.0
        while v < -1e-9:
            if m <= m_dry:
                return -1e6, -1, -1
            a_net = T / m - g
            if a_net <= 0:
                return -1e6, -1, -1
            # Compute dt_step to not overshoot v=0
            dt_step = min(dt, -v / a_net if a_net > 0 else dt)
            v_next = v + a_net * dt_step
            h_next = h + v * dt_step + 0.5 * a_net * dt_step**2
            m_next = m - (T / v_e) * dt_step
            if v_next >= -1e-9:
                # Interpolate
                frac = -v / (v_next - v)
                dt_frac = frac * dt_step
                h_v0 = h + v * dt_frac + 0.5 * a_net * dt_frac**2
                m_v0 = m - (T / v_e) * dt_frac
                t_v0 = t + dt_frac
                return h_v0, t_v0, m_v0
            h = h_next
            v = v_next
            m = m_next
            t += dt_step
        return h, t, m  # Rare case

    def f(t_coast):
        if t_coast >= t_max:
            return -1e6
        v_b = v0 - g * t_coast
        h_b = h0 + v0 * t_coast - 0.5 * g * t_coast**2
        if h_b <= 0:
            return -1e6
        return get_projected_min_h(h_b, v_b, m0, T, v_e, g)[0]

    # Check if possible
    if f(0) <= 0:
        print("Cannot land: even immediate full thrust cannot stop in time.")
    else:
        # Bisection to find t_coast where f(t_coast) = 0
        low = 0.0
        high = t_max
        for _ in range(100):  # Sufficient for precision
            mid = (low + high) / 2
            if f(mid) > 0:
                low = mid
            else:
                high = mid
        t_coast = (low + high) / 2
        v_b = v0 - g * t_coast
        h_b = h0 + v0 * t_coast - 0.5 * g * t_coast**2
        h_v0, t_burn, m_final = get_projected_min_h(h_b, v_b, m0, T, v_e, g)
        if abs(h_v0) > 0.1:  # Tolerance
            print("Cannot land precisely: check parameters or increase precision.")
        else:
            fuel_used = m0 - m_final
            if fuel_used > fuel_mass:
                print(f"Not enough fuel: requires {fuel_used:.2f} kg, but only {fuel_mass:.2f} kg available.")
            else:
                print(f"Can land optimally using {fuel_used:.2f} kg of fuel.")
                print(f"Coast for {t_coast:.2f} seconds, then full thrust for {t_burn:.2f} seconds.")
                # Simulate and plot the trajectory numerically for accuracy
                dt = 0.001
                times = []
                heights = []
                velocities = []
                thrusts = []  # thrust acceleration (m/s^2) - kept for static plot
                masses = []
                # Coast phase
                t = 0.0
                while t < t_coast:
                    height = h0 + v0 * t - 0.5 * g * t**2
                    velocity = v0 - g * t
                    times.append(t)
                    heights.append(height)
                    velocities.append(velocity)
                    thrusts.append(0)
                    masses.append(m0)
                    t += dt
                # Burn phase
                h = h_b
                v = v_b
                m = m0
                t = t_coast
                while t < t_coast + t_burn + dt:  # Slight overrun to ensure
                    times.append(t)
                    heights.append(h)
                    velocities.append(v)
                    thrust_accel = T / m if m > m_dry else 0
                    thrusts.append(thrust_accel)
                    masses.append(m)
                    a_net = thrust_accel - g
                    dt_step = min(dt, t_coast + t_burn - t) if t_coast + t_burn - t > 0 else dt
                    h_next = h + v * dt_step + 0.5 * a_net * dt_step**2
                    v_next = v + a_net * dt_step
                    m_next = m - (T / v_e) * dt_step if thrust_accel > 0 else m
                    h = h_next
                    v = v_next
                    m = m_next
                    t += dt_step
                # Verify landing
                if abs(heights[-1]) < 1e-1 and abs(velocities[-1]) < 1e-1:
                    print("Simulation confirms soft landing.")
                else:
                    print(f"Simulation landing conditions: h={heights[-1]:.2f}, v={velocities[-1]:.2f} (minor error due to dt).")
                # Static plots (optional, kept for reference)
                fig_static, axs = plt.subplots(4, 1, figsize=(8, 12))
                axs[0].plot(times, heights)
                axs[0].set_ylabel('Height (m)')
                axs[0].grid(True)
                axs[1].plot(times, velocities)
                axs[1].set_ylabel('Velocity (m/s)')
                axs[1].grid(True)
                axs[2].plot(times, thrusts)
                axs[2].set_ylabel('Thrust accel (m/s^2)')
                axs[2].grid(True)
                axs[3].plot(times, masses)
                axs[3].set_ylabel('Mass (kg)')
                axs[3].set_xlabel('Time (s)')
                axs[3].grid(True)
                plt.tight_layout()
                plt.show()

                # animation
                # configuration for sprite (exact same as advanced 2D version)
                rocket_height_vis = 95.0
                fuel_tank_height_vis = rocket_height_vis / 2
                dry_cm_vis = rocket_height_vis / 2
                fuel_cm_vis = fuel_tank_height_vis / 2
                m_dry_anim = dry_mass
                T_max_vis = T  # constant thrust force for throttle & flame scaling

                def calculate_com_and_I_anim(curr_mass):
                    fuel_remaining = max(curr_mass - m_dry_anim, 0.0)
                    com_from_nozzle = (m_dry_anim * dry_cm_vis + fuel_remaining * fuel_cm_vis) / curr_mass if curr_mass > 0 else 0.0
                    I_z = 0.0  # dummy (not used in animation)
                    return com_from_nozzle, I_z

                # Prepare data arrays
                sim_times = np.array(times)
                y_nozzle_sim = np.array(heights)
                coms_sim = np.array([calculate_com_and_I_anim(m)[0] for m in masses])
                y_coms_sim = y_nozzle_sim + coms_sim
                num_sim = len(sim_times)

                states = np.column_stack((
                    np.zeros(num_sim),          # x
                    y_coms_sim,                 # y_com
                    np.zeros(num_sim),          # vx
                    np.array(velocities),       # vy
                    np.zeros(num_sim),          # theta (rad)
                    np.zeros(num_sim),          # omega (rad)
                    np.array(masses)            # mass
                ))

                thrust_forces = np.array([T if th > 0 else 0.0 for th in thrusts])  # constant force N
                controls = np.column_stack((
                    thrust_forces,
                    np.zeros(num_sim)           # gimbal (rad)
                ))

                tf = sim_times[-1] if len(sim_times) > 0 else 0.0

                # Helper functions
                def rotate_point(x, y, pivot_x, pivot_y, angle_deg):
                    angle_rad = np.radians(angle_deg)
                    nx, ny = x - pivot_x, y - pivot_y
                    s, c = np.sin(-angle_rad), np.cos(-angle_rad)
                    rx = nx * c - ny * s
                    ry = nx * s + ny * c
                    return rx + pivot_x, ry + pivot_y

                def get_rotated_points(points_list, pivot, angle):
                    return [rotate_point(p[0], p[1], pivot[0], pivot[1], angle) for p in points_list]

                # Animation (exact sprite, telemetry, hold times, flame scaling, etc. from advanced 2D)
                def animate_results(states, controls, tf, sim_times):
                    N = states.shape[0] - 1
                    min_flame_length = 15.0
                    base_flame_length = 35
                    fps = 30
                    hold_time_start = 2.0
                    hold_time_end = 2.0
                    num_hold_start = int(hold_time_start * fps)
                    num_hold_end = int(hold_time_end * fps)
                    num_anim_frames = int(tf * fps) + 1
                    anim_times = np.linspace(0, tf, num_anim_frames)
                    dt_anim = anim_times[1] - anim_times[0] if num_anim_frames > 1 else 0

                    # Interpolate (preserves exact 1D trajectory)
                    x_coms = interp1d(sim_times, states[:, 0])(anim_times)
                    y_coms = interp1d(sim_times, states[:, 1])(anim_times)
                    vxs = interp1d(sim_times, states[:, 2])(anim_times)
                    vys = interp1d(sim_times, states[:, 3])(anim_times)
                    angles = np.rad2deg(interp1d(sim_times, states[:, 4])(anim_times))
                    omegas = np.rad2deg(interp1d(sim_times, states[:, 5])(anim_times))
                    masses = interp1d(sim_times, states[:, 6])(anim_times)
                    thrusts = interp1d(sim_times, controls[:, 0])(anim_times)
                    gimbals = np.rad2deg(interp1d(sim_times, controls[:, 1])(anim_times))

                    thrusts[-1] = 0
                    gimbals[-1] = 0

                    coms = np.array([calculate_com_and_I_anim(m)[0] for m in masses])
                    I_zs = np.zeros_like(coms)

                    # Add start hold
                    x_coms = np.concatenate((np.full(num_hold_start, x_coms[0]), x_coms))
                    y_coms = np.concatenate((np.full(num_hold_start, y_coms[0]), y_coms))
                    vxs = np.concatenate((np.full(num_hold_start, vxs[0]), vxs))
                    vys = np.concatenate((np.full(num_hold_start, vys[0]), vys))
                    angles = np.concatenate((np.full(num_hold_start, angles[0]), angles))
                    omegas = np.concatenate((np.full(num_hold_start, omegas[0]), omegas))
                    masses = np.concatenate((np.full(num_hold_start, masses[0]), masses))
                    thrusts = np.concatenate((np.full(num_hold_start, thrusts[0]), thrusts))
                    gimbals = np.concatenate((np.full(num_hold_start, gimbals[0]), gimbals))
                    coms = np.concatenate((np.full(num_hold_start, coms[0]), coms))
                    I_zs = np.concatenate((np.full(num_hold_start, I_zs[0]), I_zs))

                    # Add end hold (velocities forced to 0, thrust=0)
                    x_coms = np.append(x_coms, np.full(num_hold_end, x_coms[-1]))
                    y_coms = np.append(y_coms, np.full(num_hold_end, y_coms[-1]))
                    vxs = np.append(vxs, np.full(num_hold_end, 0.0))
                    vys = np.append(vys, np.full(num_hold_end, 0.0))
                    angles = np.append(angles, np.full(num_hold_end, angles[-1]))
                    omegas = np.append(omegas, np.full(num_hold_end, 0.0))
                    masses = np.append(masses, np.full(num_hold_end, masses[-1]))
                    thrusts = np.append(thrusts, np.full(num_hold_end, 0.0))
                    gimbals = np.append(gimbals, np.full(num_hold_end, gimbals[-1]))
                    coms = np.append(coms, np.full(num_hold_end, coms[-1]))
                    I_zs = np.append(I_zs, np.full(num_hold_end, I_zs[-1]))

                    anim_times_start = np.linspace(-hold_time_start, -dt_anim, num_hold_start)
                    anim_times_end = np.linspace(tf + dt_anim, tf + hold_time_end, num_hold_end)
                    anim_times = np.concatenate((anim_times_start, anim_times, anim_times_end))
                    num_anim_frames += num_hold_start + num_hold_end

                    # Figure setup
                    margin = rocket_height_vis * 1.5
                    x_min = min(x_coms) - margin
                    x_max = max(x_coms) + margin
                    y_min = -rocket_height_vis
                    y_max = max(y_coms) + margin
                    x_span = x_max - x_min
                    y_span = y_max - y_min
                    if x_span < y_span:
                        extra = (y_span - x_span) / 2
                        x_min -= extra
                        x_max += extra
                    elif y_span < x_span:
                        extra = (x_span - y_span) / 2
                        y_min -= extra
                        y_max += extra

                    fig, (ax, ax_info) = plt.subplots(1, 2, figsize=(12, 8), gridspec_kw={'width_ratios': [2, 1]})
                    ax.set_xlim(x_min, x_max)
                    ax.set_ylim(y_min, y_max)
                    ax.set_aspect('equal')
                    ax.axis('off')

                    # Gradient sky
                    custom_cmap = LinearSegmentedColormap.from_list("sky", ["#102C57", "#000000"])
                    ax.imshow(np.linspace(0, 1, 100).reshape(-1, 1), extent=[x_min, x_max, y_min, y_max],
                              origin='lower', cmap=custom_cmap, aspect='auto')

                    # Ground
                    ax.axhspan(y_min, 0, color="#888888", zorder=1)
                    ax.axhline(0, color="#B1B1B1", linewidth=2, zorder=2)

                    # Landing pad (centered)
                    pad = Rectangle((-20, -5), 40, 5, fc="gray", ec="black", zorder=3)
                    ax.add_patch(pad)

                    # Rocket sprites
                    body_patch = Polygon([[0,0]], fc="white", ec="black", zorder=11)
                    nose_patch = Polygon([[0,0]], fc="white", ec="black", zorder=12)
                    engine_patch = Polygon([[0,0]], fc="#D0D0D0", ec="black", zorder=10)
                    flame_patch = Polygon([[0,0]], fc="orange", ec="red", lw=1.5, zorder=9, visible=False)
                    ax.add_patch(body_patch)
                    ax.add_patch(nose_patch)
                    ax.add_patch(engine_patch)
                    ax.add_patch(flame_patch)

                    # Initial velocity arrow (shown only in start hold)
                    arrow_length = abs(x_max - x_min) * 0.1
                    vel_arrow = FancyArrowPatch(
                        (0,0), (0,0),
                        arrowstyle="simple, head_width=9, head_length=9, tail_width=4",
                        mutation_scale=1.5, facecolor='cyan', edgecolor='black',
                        linewidth=1.5, zorder=20, visible=False, label="Initial Velocity"
                    )
                    ax.add_patch(vel_arrow)

                    # Bottom plot labels
                    pos_vel_text1 = ax.text(0.12, 0.06, '', transform=ax.transAxes, color='white',
                                            fontsize=13, family='monospace', fontweight='bold', va='bottom', ha='left')
                    pos_vel_text2 = ax.text(0.12, 0.02, '', transform=ax.transAxes, color='white',
                                            fontsize=13, family='monospace', fontweight='bold', va='bottom', ha='left')

                    # Telemetry panel
                    ax_info.set_facecolor('white')
                    ax_info.axis('off')
                    left_bound = -0.2
                    fontsize = 15
                    txt_alt = ax_info.text(left_bound, 0.8, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
                    txt_downrange = ax_info.text(left_bound, 0.7, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
                    txt_theta = ax_info.text(left_bound, 0.6, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
                    txt_omega = ax_info.text(left_bound, 0.5, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
                    txt_gimbal = ax_info.text(left_bound, 0.4, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
                    txt_thrust = ax_info.text(left_bound, 0.3, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
                    txt_fuel = ax_info.text(left_bound, 0.2, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
                    txt_time = ax_info.text(left_bound, 0.1, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
                    ax_info.text(0.5, 0.9, "TELEMETRY", color='black', fontsize=fontsize + 4, fontweight='bold', ha="center")

                    # Raw sprite coordinates
                    body_raw = [[-7,10], [7,10], [7,80], [-7,80]]
                    nose_raw = [[-7,80], [7,80], [0,95]]
                    engine_attach = (0, 10)
                    engine_raw = [[-8.4,0], [8.4,0], [4.5,10], [-4.5,10]]
                    engine_rel = [(ex - engine_attach[0], ey - engine_attach[1]) for ex, ey in engine_raw]
                    flame_raw = [[-5.6,0], [5.6,0], [0, -base_flame_length]]
                    flame_rel = [(fx - engine_attach[0], fy - engine_attach[1]) for fx, fy in flame_raw]

                    def animate(i):
                        t_val = angles[i]
                        g_val = gimbals[i]
                        thr_val = thrusts[i]
                        cx = x_coms[i]
                        cy = y_coms[i]
                        vx_i, vy_i = vxs[i], vys[i]
                        com = coms[i]

                        # Local coordinates relative to CoM
                        body_local = [[px, py - com] for px, py in body_raw]
                        nose_local = [[px, py - com] for px, py in nose_raw]
                        engine_attach_local = (0, 10 - com)

                        body_rot = get_rotated_points(body_local, (0, 0), t_val)
                        nose_rot = get_rotated_points(nose_local, (0, 0), t_val)
                        engine_gimb_rel = get_rotated_points(engine_rel, (0, 0), g_val)
                        engine_local = [(ex + engine_attach_local[0], ey + engine_attach_local[1]) for ex, ey in engine_gimb_rel]
                        engine_rot = get_rotated_points(engine_local, (0, 0), t_val)

                        # Flame (scales with throttle, gimbals)
                        throttle = thr_val / T_max_vis if T_max_vis > 0 else 0.0
                        gamma = 0.5
                        flame_length = base_flame_length * (throttle ** gamma)
                        flame_rel_scaled = [(fx, fy * (flame_length / base_flame_length)) for fx, fy in flame_rel]
                        flame_gimb_rel = get_rotated_points(flame_rel_scaled, (0, 0), g_val)
                        flame_local = [(fx + engine_attach_local[0], fy + engine_attach_local[1]) for fx, fy in flame_gimb_rel]
                        flame_rot = get_rotated_points(flame_local, (0, 0), t_val)

                        body_patch.set_xy([(px + cx, py + cy) for px, py in body_rot])
                        nose_patch.set_xy([(px + cx, py + cy) for px, py in nose_rot])
                        engine_patch.set_xy([(px + cx, py + cy) for px, py in engine_rot])
                        flame_patch.set_xy([(px + cx, py + cy) for px, py in flame_rot])
                        flame_patch.set_visible(thr_val > 0)

                        # Telemetry (exact formatting & alignment)
                        LABEL_WIDTH = len("FUEL BURNED:") + 1
                        VALUE_WIDTH = 11
                        nozzle_alt = y_coms[i] - coms[i]
                        txt_alt.set_text(f"{'ALTITUDE:':>{LABEL_WIDTH}}{nozzle_alt:>{VALUE_WIDTH}.2f} m")
                        txt_downrange.set_text(f"{'DOWNRANGE:':>{LABEL_WIDTH}}{x_coms[i]:>{VALUE_WIDTH}.2f} m")
                        txt_theta.set_text(f"{'THETA:':>{LABEL_WIDTH}}{t_val:>{VALUE_WIDTH}.2f} °")
                        txt_omega.set_text(f"{'OMEGA:':>{LABEL_WIDTH}}{omegas[i]:>{VALUE_WIDTH}.2f} °/s")
                        txt_gimbal.set_text(f"{'GIMBAL:':>{LABEL_WIDTH}}{g_val:>{VALUE_WIDTH}.2f} °")
                        txt_thrust.set_text(f"{'THRUST:':>{LABEL_WIDTH}}{thr_val:>{VALUE_WIDTH}.0f} N")
                        txt_fuel.set_text(f"{'FUEL:':>{LABEL_WIDTH}}{masses[i] - m_dry_anim:>{VALUE_WIDTH}.2f} kg")
                        txt_time.set_text(f"{'TIME:':>{LABEL_WIDTH}}{max(0, min(anim_times[i], tf)):>{VALUE_WIDTH}.2f} s")

                        pos_vel_text1.set_text(f" X: {x_coms[i]:10.2f} m   Y: {nozzle_alt:10.2f} m")
                        pos_vel_text2.set_text(f"Vx: {vxs[i]:10.2f} m/s Vy: {vys[i]:10.2f} m/s")

                        # Velocity arrow only during start hold
                        show_vel_arrow = anim_times[i] < 0
                        speed = np.sqrt(vx_i**2 + vy_i**2)
                        if show_vel_arrow and speed > 1e-3:
                            base_x, base_y = cx, cy
                            dx = (vx_i / speed) * arrow_length
                            dy = (vy_i / speed) * arrow_length
                            vel_arrow.set_positions((base_x, base_y), (base_x + dx, base_y + dy))
                            vel_arrow.set_visible(True)
                        else:
                            vel_arrow.set_visible(False)

                        # Legend (updates dynamically)
                        legend = ax.get_legend()
                        if legend is not None:
                            legend.remove()
                        if show_vel_arrow and speed > 1e-3:
                            handles = [vel_arrow]
                            labels = ["Initial v"]
                            ax.legend(handles, labels, loc='upper right', fontsize=14, framealpha=0.7)


                        return (body_patch, nose_patch, engine_patch, flame_patch,
                                txt_alt, txt_downrange, txt_theta, txt_omega, txt_gimbal,
                                txt_thrust, txt_fuel, txt_time, pos_vel_text1, pos_vel_text2, vel_arrow)

                    ani = FuncAnimation(fig, animate, frames=range(num_anim_frames),
                                        interval=1000 // fps, blit=True)

                    print("Saving animation...")
                    writer = FFMpegWriter(fps=fps, bitrate=2500)
                    file_name = "rocket_landing_1D"
                    ani.save(f"../results/{file_name}.mp4", writer=writer)
                    print(f"Animation saved as 'results/{file_name}.mp4'")
                    print("Done.")
                    plt.show()

                # Launch the new animation
                animate_results(states, controls, tf, sim_times)

As visible in the plots, when we assumed constant mass earlier, we saw a the classic bang-bang behaviour predicted by the Pontryagin Maximum Principle arising in the controls. However, when we take into account the decreasing mass during fuel burn, the Thrust force $T$ stays constant, yet, according to Newton,
$$a = \frac{T}{m}.$$

Hence, the Thrust acceleration increases over time, but the control is still bang-bang (constant full throttle after the switching time).

# Attitude Control System (Rotation)

Consider a rocket floating in outer space (in other words, assume the rocket is in a vacuum, with no gravity, no aerodynamic forces). Assume also that the reference frame is fixed to the center of mass of the rocket, such that we only consider the rotational orientation of the rocket in the xy-plane, i.e. the angle between the nose of the rocket and the y-axis (vertical). Assume the thrusters of the rocket are attached to a gimbal which can tilt within $\phi \in [-10\degree, 10\degree]$.

Given initial angle $\theta_0$, an initial angular velocity $\dot{\theta} = \omega$, a target angle $\theta_t$, as well as the dimensions of the rocket (dry mass $m_d$, fuel mass $m_f$, maximum possible thrust $T$ and exhaust velocity $v_e$), is it possible to steer the rocket into the target position $\theta_t$ with zero angular velocity?

If so, which control sequence does so in minimal time $t^*$? (The fuel optimal case is not so interesting here; given the vacuum conditions, it would be a tiny nudge, which causes the rocket to start rotating ever so slowly, and since no external forces act on it, it would spin very slowly until it reached the target position)

This is a common 1D control problem, with the added challenge of the circular freedom of the state: Any initial angle $\theta_0$ can be steered into the target $\theta_t$ in two ways (clockwise and counterclockwise). The tricky part is deciding which is faster, given the inital angle $\theta_0$, the initial angular velocity $\omega$ and the target angle $\theta_t$.


However, we take into account that the mass changes as fuel is burned, so $\dot{m} \not= 0$. We model the time dependent mass through the Tsiolkovsky equation, $$m(t + \Delta t) = m(t) + \dot{m} \cdot \Delta t = m(t) - \frac{T}{v_e} \Delta t,$$ with discrete time steps $\Delta t$ and a mass burn rate of $\dot{m} = -\frac{T}{v_e}$.

The torque stemming from the tilted gimbal is $$\tau = T \cdot sin(\phi)\cdot l,$$ where $T$ is the Thrust, $\phi$ is the angle of the gimbal, and $l$ is the lever arm, which is the distance from the center of mass of the rocket to the nozzle, so roughly half the height of the rocket, $h/2$. 

The angular acceleration is given by $$\alpha(t) = \frac{\tau}{I(t)} = \frac{12\cdot T \cdot sin(\phi(t))\cdot l}{m(t)\cdot h^2}$$ where $m(t)$ is the current total mass (dry mass + fuel mass) at time $t$. Using discrete time steps $\Delta t$ and Euler integration, we obtain $$\omega(t + \Delta t) = \omega(t) + \alpha(t) \cdot \Delta t$$ as well as $$\theta(t + \Delta t) = \theta(t) + \omega(t) \cdot \Delta t.$$ 
As we will see later, the time-optimal controls are (as predicted by theory) bang-bang controls, which means that the control parameter $\phi(t)$ takes on only the extreme points of the control parameters, $\{-10\degree, 10\degree\}$ as values, with near instant switches between the states. The thrust will also be bang-bang, so either full thrust or no thrust at all. This may not be fuel-efficient, but it is time-optimal.

The guidance law is a smoothed bang-bang / near-time-optimal controller approximating a parabolic switching curve in the phase plane $  (\theta, \omega)  $.
It tries to follow
$$\omega_\text{command} \approx \text{sign}(\text{error}) \sqrt{2 \cdot \alpha_\text{max} \cdot |\text{error}|}$$
which is the equation of the minimum-time switching curve for bounded acceleration.

Note that because inertia decreases over time, the true time-optimal gimbal profile is more complex than pure bang-bang with a fixed switching curve; the simulation uses a constant-acceleration approximation that is close to optimal for short maneuvers.

In [ ]:
import numpy as np
import sys
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Polygon, FancyArrowPatch
from matplotlib.colors import LinearSegmentedColormap

# ===========================================================================
# Geometry helpers
# ===========================================================================

def wrap_angle(a):
    """Wrap angle into (-180, 180] degrees."""
    return (a + 180) % 360 - 180

def rotate_point(x, y, pivot_x, pivot_y, angle_deg):
    angle_rad = np.radians(angle_deg)
    nx, ny = x - pivot_x, y - pivot_y
    s, c   = np.sin(-angle_rad), np.cos(-angle_rad)
    return nx*c - ny*s + pivot_x, nx*s + ny*c + pivot_y

def get_rotated_points(pts, pivot, angle):
    return [rotate_point(p[0], p[1], pivot[0], pivot[1], angle) for p in pts]

# ===========================================================================
# Physics
# ===========================================================================

def calculate_com_and_I(curr_mass, fuel_remaining,
                         dry_mass, rocket_height, fuel_tank_height,
                         dry_cm_from_nozzle, fuel_cm_from_nozzle):
    """Return (CoM from nozzle [m], axial moment of inertia [kg·m²])."""
    com = (dry_mass * dry_cm_from_nozzle + fuel_remaining * fuel_cm_from_nozzle) / curr_mass

    I_dry  = (1/12)*dry_mass*rocket_height**2     + dry_mass *(dry_cm_from_nozzle  - com)**2
    I_fuel = (1/12)*fuel_remaining*fuel_tank_height**2 + fuel_remaining*(fuel_cm_from_nozzle - com)**2
    return com, I_dry + I_fuel

# ===========================================================================
# User inputs
# ===========================================================================

theta_0      = wrap_angle(float(input("Initial angle (deg): ")))
theta_target = wrap_angle(float(input("Target angle (deg): ")))
omega_0      = float(input("Initial angular velocity (deg/s): "))
dry_mass     = float(input("Dry mass (kg): "))
fuel_mass    = float(input("Fuel mass (kg): "))
thrust_force = float(input("Thrust Force (N): "))
v_e          = float(input("Exhaust velocity (m/s): "))

render_both = False
resp = input("Render both views side-by-side? (y/n): ").strip().lower()
if resp in ('y', 'yes', '1', 'true'):
    render_both         = True
    include_translation = True
    print("--> Running BOTH modes side-by-side: Rotation-Only (left) + Full Translation (right)")
else:
    resp = input("Include translation (full 6DOF motion)? (y/n): ").strip().lower()
    include_translation = resp in ('y', 'yes', '1', 'true')
    print(f"--> Running {'WITH TRANSLATION' if include_translation else 'ROTATION ONLY'} mode.")

# ===========================================================================
# Rocket specs (geometry / control limits)
# ===========================================================================

rocket_height       = 70
fuel_tank_height    = rocket_height / 2
dry_cm_from_nozzle  = rocket_height / 2
fuel_cm_from_nozzle = fuel_tank_height / 2
gimbal_limit        = 10.0   # deg
max_gimbal_speed    = 360.0  # deg/s

# Drawing geometry (body-frame, origin at geometric centre)
BODY_RAW      = [[-7, -35], [7, -35], [7, 35], [-7, 35]]
NOSE_RAW      = [[-7,  35], [7,  35], [0, 50]]
ENGINE_ATTACH = (0, -34)
ENGINE_RAW    = [[-8.4, -45], [8.4, -45], [4.5, -34], [-4.5, -34]]
FLAME_RAW     = [[-5.6, -45], [5.6, -45], [0,   -70]]
# Pre-compute engine/flame relative to their attach point (used every frame)
ENGINE_REL = [(ex - ENGINE_ATTACH[0], ey - ENGINE_ATTACH[1]) for ex, ey in ENGINE_RAW]
FLAME_REL  = [(fx - ENGINE_ATTACH[0], fy - ENGINE_ATTACH[1]) for fx, fy in FLAME_RAW]

# Telemetry formatting constants
LW, VW = len("FUEL BURNED:") + 1, 11

# ===========================================================================
# Controllability pre-check
# ===========================================================================

initial_mass = dry_mass + fuel_mass
initial_com, initial_I = calculate_com_and_I(
    initial_mass, fuel_mass, dry_mass, rocket_height,
    fuel_tank_height, dry_cm_from_nozzle, fuel_cm_from_nozzle)

# Maximum achievable angular acceleration (deg/s²)
max_alpha_possible = np.degrees(
    thrust_force * np.sin(np.radians(gimbal_limit)) * initial_com / initial_I)

fuel_burn_rate = thrust_force / v_e

# Check 1: can we stop the initial spin?
stopping_time = abs(omega_0) / max_alpha_possible if max_alpha_possible > 0 else 0
if fuel_burn_rate * stopping_time > fuel_mass:
    sys.exit("ABORT: Not enough fuel to stop the initial rotation.")

# Choose short vs long path to target
short_error   = wrap_angle(theta_target - theta_0)
stopping_dist = omega_0**2 / (2*max_alpha_possible) if max_alpha_possible > 0 else 9_999_999
use_wrap      = True
effective_target = theta_target

if np.sign(omega_0) != np.sign(short_error) and stopping_dist > abs(short_error):
    # Momentum carries us past the short path — long path is cheaper
    use_wrap         = False
    effective_target = theta_0 + (short_error - 360 * np.sign(short_error))
    print(f"Using long path (effective target: {effective_target:.2f}°)")
else:
    print("Using short path.")

# Check 2: enough fuel for the full manoeuvre?
maneuver_time = 2 * np.sqrt(abs(effective_target - theta_0) / max_alpha_possible) \
                if max_alpha_possible > 0 else 0
fuel_stop     = (fuel_burn_rate * abs(omega_0) / max_alpha_possible
                 if np.sign(omega_0) != np.sign(effective_target - theta_0) and max_alpha_possible > 0
                 else 0)
if fuel_burn_rate * maneuver_time + fuel_stop > fuel_mass:
    sys.exit("ABORT: Not enough fuel to reach the target angle.")

print("System is controllable. Proceeding to simulation...")

# ===========================================================================
# Simulation loop
# ===========================================================================

dt      = 0.01
BOUNDARY = 0.2   # bang-bang boundary layer width (deg)

# State
curr_theta  = theta_0
curr_omega  = omega_0
curr_gimbal = 0.0
curr_mass   = dry_mass + fuel_mass
x_com = y_com = vx = vy = 0.0

# Time-series storage (always allocate all channels; unused ones stay at 0)
times, angles, unwrapped_angles = [], [], []
omegas, gimbals, masses         = [], [], []
thrusts, torques, coms, Is      = [], [], [], []
x_coms, y_coms, vxs, vys       = [], [], [], []

t        = 0.0
settled  = False
running  = True

while running and t < 40:
    fuel_remaining = max(curr_mass - dry_mass, 0.0)
    com, I = calculate_com_and_I(
        curr_mass, fuel_remaining, dry_mass, rocket_height,
        fuel_tank_height, dry_cm_from_nozzle, fuel_cm_from_nozzle)
    lever_arm = com
    max_alpha = np.degrees(thrust_force * np.sin(np.radians(gimbal_limit)) * lever_arm / I)

    # --- Bang-bang with boundary layer ---
    error        = wrap_angle(effective_target - curr_theta) if use_wrap else effective_target - curr_theta
    target_omega = np.sign(error) * np.sqrt(2 * max_alpha * abs(error))
    switch_err   = target_omega - curr_omega

    if abs(switch_err) > BOUNDARY:
        target_gimbal_cmd = -np.sign(switch_err) * gimbal_limit
    else:
        target_gimbal_cmd = -(switch_err / BOUNDARY) * gimbal_limit

    # Settled check
    active_thrust = True
    if abs(error) < 0.2 and abs(curr_omega) < 0.2:
        target_gimbal_cmd = 0.0
        curr_omega        = 0.0
        active_thrust     = False
        settled           = True

    # Gimbal slew-rate limit
    gimbal_err   = target_gimbal_cmd - curr_gimbal
    if abs(gimbal_err) > 0.01:
        step         = np.sign(gimbal_err) * max_gimbal_speed * dt
        curr_gimbal += np.clip(step, -abs(gimbal_err), abs(gimbal_err))

    # --- Physics ---
    current_thrust = thrust_force if active_thrust else 0.0

    if include_translation or render_both:
        thrust_rad  = np.radians(curr_theta + curr_gimbal)
        if curr_mass > dry_mass + 1e-6:
            vx += current_thrust * np.sin(thrust_rad) / curr_mass * dt
            vy += current_thrust * np.cos(thrust_rad) / curr_mass * dt
        x_com += vx * dt
        y_com += vy * dt

    torque      = -thrust_force * np.sin(np.radians(curr_gimbal)) * lever_arm
    curr_omega += np.degrees(torque / I) * dt
    curr_theta += curr_omega * dt

    if active_thrust:
        curr_mass = max(curr_mass - current_thrust / v_e * dt, dry_mass)

    # --- Record ---
    times.append(t);             angles.append(wrap_angle(curr_theta))
    unwrapped_angles.append(curr_theta)
    omegas.append(curr_omega);   gimbals.append(curr_gimbal)
    masses.append(curr_mass);    thrusts.append(current_thrust)
    torques.append(torque);      coms.append(com);  Is.append(I)
    x_coms.append(x_com);        y_coms.append(y_com)
    vxs.append(vx);              vys.append(vy)

    t += dt
    if (settled and abs(curr_gimbal) < 0.01) or curr_mass <= dry_mass:
        running = False

final_time = t   # manoeuvre end time (before hold)

# Post-settle hold (1.5 s of frozen state)
for _ in range(int(1.5 / dt)):
    times.append(times[-1] + dt)
    angles.append(wrap_angle(curr_theta));  unwrapped_angles.append(curr_theta)
    omegas.append(0.0);  gimbals.append(0.0)
    masses.append(masses[-1]);  thrusts.append(0.0);  torques.append(0.0)
    coms.append(coms[-1]);      Is.append(Is[-1])
    x_coms.append(x_coms[-1]); y_coms.append(y_coms[-1])
    vxs.append(vxs[-1]);        vys.append(vys[-1])

print(f"Final stopping time: {final_time:.2f} s")

# ===========================================================================
# Diagnostic plots
# ===========================================================================

if include_translation or render_both:
    plot_data = [angles, omegas, gimbals, masses, thrusts, torques,
                 coms, Is, x_coms, y_coms, vxs, vys]
    plot_labels = ['Angle (deg)', 'Ω (deg/s)', 'Gimbal (deg)', 'Mass (kg)',
                   'Thrust (N)', 'Torque (N·m)', 'CoM from nozzle (m)',
                   'Moment of Inertia (kg·m²)', 'X pos (m)', 'Y pos (m)', 'Vx (m/s)', 'Vy (m/s)']
    plot_colors = ['cyan','lime','orange','red','blue','purple',
                   'magenta','gold','teal','navy','brown','olive']
    diag_title  = "Post-Flight Analysis (Full 6DOF Simulation)"
else:
    plot_data   = [angles, omegas, gimbals, masses, thrusts, torques, coms, Is]
    plot_labels = ['Angle (deg)', 'Ω (deg/s)', 'Gimbal (deg)', 'Mass (kg)',
                   'Thrust (N)', 'Torque (N·m)', 'CoM from nozzle (m)', 'Moment of Inertia (kg·m²)']
    plot_colors = ['cyan','lime','orange','red','blue','purple','magenta','gold']
    diag_title  = "Post-Flight Analysis"

fig_diag, axs = plt.subplots(len(plot_data), 1, figsize=(10, 2*len(plot_data)+2), sharex=True)
plt.subplots_adjust(hspace=0.28)
for i, ax in enumerate(axs):
    ax.plot(times, plot_data[i], color=plot_colors[i])
    ax.set_ylabel(plot_labels[i])
    ax.set_facecolor("#ffffff")
    ax.grid(True, alpha=0.18)
    ax.set_xlim(-0.4, times[-1] + 0.6)
    ax.axvline(final_time - dt, color="red", linestyle="--", alpha=0.5, lw=1.2)
axs[0].axhline(theta_target, color='black', ls='--', alpha=0.6, label="target")
axs[0].legend(fontsize=9)
axs[7].axhline(Is[0],  color='darkgreen', ls='--', alpha=0.5, label=f"initial I = {Is[0]:.0f}")
axs[7].axhline(Is[-1], color='darkred',   ls='--', alpha=0.5, label=f"final I = {Is[-1]:.0f}")
axs[7].legend(fontsize=9, loc='upper right')
axs[-1].set_xlabel("Time (s)")
plt.suptitle(diag_title, fontsize=16, y=0.995)
plt.show()

# ===========================================================================
# Animation helpers (shared by all three modes)
# ===========================================================================

def _sky_bg(ax, extent):
    cmap = LinearSegmentedColormap.from_list("sky", ["#102C57", "#000000"])
    ax.imshow(np.linspace(0, 1, 100).reshape(-1, 1),
              extent=extent, origin='lower', cmap=cmap, aspect='auto')

def _add_rocket_patches(ax):
    """Create the four body patches, add them to ax, return them."""
    bp = Polygon([(0,0)], fc="white",   ec="black", zorder=11)
    np_ = Polygon([(0,0)], fc="white",  ec="black", zorder=12)
    ep = Polygon([(0,0)], fc="#D0D0D0", ec="black", zorder=10)
    fp = Polygon([(0,0)], fc="orange",  ec="red",   lw=1.5, zorder=9)
    for p in [bp, np_, ep, fp]:
        ax.add_patch(p)
    return bp, np_, ep, fp

def _set_rocket_geometry_rot(patches, theta_deg, gimbal_deg, thrust):
    """Update patches for rotation-only view (rocket fixed at origin)."""
    bp, np_, ep, fp = patches
    eng_r  = get_rotated_points(ENGINE_RAW, ENGINE_ATTACH, gimbal_deg)
    flm_r  = get_rotated_points(FLAME_RAW,  ENGINE_ATTACH, gimbal_deg)
    bp.set_xy( get_rotated_points(BODY_RAW, (0,0), theta_deg))
    np_.set_xy(get_rotated_points(NOSE_RAW, (0,0), theta_deg))
    ep.set_xy( get_rotated_points(eng_r,    (0,0), theta_deg))
    fp.set_xy( get_rotated_points(flm_r,    (0,0), theta_deg))
    fp.set_visible(thrust > 0)

def _set_rocket_geometry_trans(patches, theta_deg, gimbal_deg, thrust, cx, cy, local_com_y):
    """Update patches for translation view (rocket moves with CoM)."""
    bp, np_, ep, fp = patches
    # Shift body/nose so CoM sits at origin, then rotate, then translate to world pos
    body_local = [(bx, by - local_com_y) for bx, by in BODY_RAW]
    nose_local = [(nx, ny - local_com_y) for nx, ny in NOSE_RAW]
    ea_local   = (ENGINE_ATTACH[0], ENGINE_ATTACH[1] - local_com_y)
    eng_g  = get_rotated_points(ENGINE_REL, (0,0), gimbal_deg)
    flm_g  = get_rotated_points(FLAME_REL,  (0,0), gimbal_deg)
    eng_l  = [(ex + ea_local[0], ey + ea_local[1]) for ex, ey in eng_g]
    flm_l  = [(fx + ea_local[0], fy + ea_local[1]) for fx, fy in flm_g]
    def _world(pts): return [(px+cx, py+cy) for px,py in get_rotated_points(pts, (0,0), theta_deg)]
    bp.set_xy(_world(body_local));  np_.set_xy(_world(nose_local))
    ep.set_xy(_world(eng_l));       fp.set_xy(_world(flm_l))
    fp.set_visible(thrust > 0)

def _omega_arrow(ax, arrow_patch, tip_x, tip_y, omega_sign, arrow_len, extra_patches=None):
    """Point the angular-velocity arrow tangentially at the nose tip."""
    r = np.sqrt(tip_x**2 + tip_y**2) or 1
    dx = -omega_sign * (-tip_y / r) * arrow_len
    dy = -omega_sign * (tip_x  / r) * arrow_len
    arrow_patch.set_positions((tip_x, tip_y), (tip_x+dx, tip_y+dy))
    arrow_patch.set_visible(True)

def _update_legend(ax, lines, labels):
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()
    ax.legend(lines, labels, loc='upper right', fontsize=15, framealpha=0.7)

ANIM_FRAMES = [0]*120 + list(range(0, len(times), 2))

# ===========================================================================
# Animation — Rotation-only mode
# ===========================================================================

def _build_anim_rotation():
    fig_size = rocket_height + 40
    fig, (ax, ax_info) = plt.subplots(1, 2, figsize=(12, 8), gridspec_kw={'width_ratios': [2,1]})
    ax.set_xlim(-fig_size, fig_size);  ax.set_ylim(-fig_size, fig_size)
    ax.set_aspect('equal');             ax.axis('off')
    _sky_bg(ax, [-fig_size, fig_size, -fig_size, fig_size])
    patches = _add_rocket_patches(ax)

    rad1 = np.radians(theta_0);  rad2 = np.radians(theta_target)
    init_line, = ax.plot([0, 80*np.sin(rad1)], [0, 80*np.cos(rad1)],
                         color='red',   ls='--', alpha=0.6, zorder=5)
    tgt_line,  = ax.plot([0, 80*np.sin(rad2)], [0, 80*np.cos(rad2)],
                         color='green', ls='--', alpha=0.7, zorder=5)
    arr = FancyArrowPatch((0,0),(0,0), mutation_scale=20, color='yellow', visible=False)
    ax.add_patch(arr)
    ax.text(0.5, 1.02, f"Initial ω₀ = {omega_0:.2f} °/s",
            family='monospace', fontweight='bold', ha='center', va='bottom',
            color='black', fontsize=12, transform=ax.transAxes)

    ax_info.set_facecolor('white');  ax_info.axis('off')
    lb, fs = -0.2, 15
    ax_info.text(0.5, 0.9, "LIVE TELEMETRY", color='black', fontsize=fs+4, fontweight='bold', ha="center")
    tx = {k: ax_info.text(lb, y, '', color='black', fontsize=fs, family='monospace', fontweight='bold')
          for k, y in [('angle',0.8),('omega',0.7),('gimbal',0.6),('thrust',0.5),
                       ('fuel',0.4),('burned',0.3),('torque',0.2),('time',0.1)]}

    def animate(i):
        t_val, g_val = angles[i], gimbals[i]
        _set_rocket_geometry_rot(patches, t_val, g_val, thrusts[i])

        tx['angle'].set_text( f"{'BODY TILT:':>{LW}}{t_val:>{VW}.2f} [°]")
        tx['omega'].set_text( f"{'ANG VEL:':>{LW}}{omegas[i]:>{VW}.2f} [°/s]")
        tx['gimbal'].set_text(f"{'GIMBAL:':>{LW}}{g_val:>{VW}.2f} [°]")
        tx['thrust'].set_text(f"{'THRUST:':>{LW}}{thrusts[i]:>{VW}.0f} [N]")
        tx['fuel'].set_text(  f"{'FUEL LEFT:':>{LW}}{masses[i]-dry_mass:>{VW}.2f} [kg]")
        tx['burned'].set_text(f"{'FUEL USED:':>{LW}}{fuel_mass-(masses[i]-dry_mass):>{VW}.2f} [kg]")
        tx['torque'].set_text(f"{'TORQUE:':>{LW}}{torques[i]:>{VW}.0f} [N·m]")
        tx['time'].set_text(  f"{'TIME:':>{LW}}{min(times[i], final_time):>{VW}.2f} [s]")

        show_arrow = times[i] < dt and abs(omega_0) > 1e-6
        tip = get_rotated_points([(0, 50)], (0,0), t_val)[0]
        if show_arrow:
            _omega_arrow(ax, arr, tip[0], tip[1], np.sign(omega_0), 20)
            _update_legend(ax, [init_line, tgt_line, arr],
                           ["Initial Angle", "Target Angle", r"Initial $\omega$"])
        else:
            arr.set_visible(False)
            _update_legend(ax, [init_line, tgt_line], ["Initial Angle", "Target Angle"])

        return (*patches, *tx.values(), arr)

    return fig, animate

# ===========================================================================
# Animation — Translation mode
# ===========================================================================

def _build_anim_translation():
    max_disp = max(max(abs(x) for x in x_coms), max(abs(y) for y in y_coms), 1)
    fig_size = max(1.75 * max_disp, rocket_height + 30)
    arrow_len = 0.2 * fig_size

    fig, (ax, ax_info) = plt.subplots(1, 2, figsize=(12, 8), gridspec_kw={'width_ratios': [2,1]})
    ax.set_xlim(-fig_size, fig_size);  ax.set_ylim(-fig_size+25, fig_size)
    ax.set_aspect('equal');             ax.axis('off')
    _sky_bg(ax, [-fig_size, fig_size, -fig_size, fig_size])
    patches = _add_rocket_patches(ax)

    rad1 = np.radians(theta_0);  rad2 = np.radians(theta_target)
    init_line, = ax.plot([0, 80*np.sin(rad1)], [0, 80*np.cos(rad1)],
                         color='red',   ls='--', alpha=0.6, zorder=5)
    tgt_line,  = ax.plot([0, 80*np.sin(rad2)], [0, 80*np.cos(rad2)],
                         color='green', ls='--', alpha=0.7, zorder=5)
    arr     = FancyArrowPatch((0,0),(0,0), mutation_scale=20, color='yellow', visible=False, zorder=20)
    vel_arr = FancyArrowPatch((0,0),(0,0), mutation_scale=20, color='cyan',   visible=False, zorder=20)
    ax.add_patch(arr);  ax.add_patch(vel_arr)
    ax.text(0.5, 1.02, f"Initial ω₀ = {omega_0:.2f} °/s",
            family='monospace', fontweight='bold', ha='center', va='bottom',
            color='black', fontsize=12, transform=ax.transAxes)

    ax_info.set_facecolor('white');  ax_info.axis('off')
    lb, fs = -0.2, 15
    ax_info.text(0.5, 0.9, "LIVE TELEMETRY", color='black', fontsize=fs+4, fontweight='bold', ha="center")
    tx = {k: ax_info.text(lb, y, '', color='black', fontsize=fs, family='monospace', fontweight='bold')
          for k, y in [('angle',0.8),('omega',0.7),('gimbal',0.6),('thrust',0.5),
                       ('fuel',0.4),('burned',0.3),('torque',0.2),('time',0.1)]}
    pv1 = ax.text(0.05, 0.06, '', transform=ax.transAxes, color='white',
                  fontsize=13, family='monospace', fontweight='bold', va='bottom', ha='left')
    pv2 = ax.text(0.05, 0.02, '', transform=ax.transAxes, color='white',
                  fontsize=13, family='monospace', fontweight='bold', va='bottom', ha='left')

    def animate(i):
        t_val   = unwrapped_angles[i]
        g_val   = gimbals[i]
        cx, cy  = x_coms[i], y_coms[i]
        local_com_y = coms[i] - dry_cm_from_nozzle
        _set_rocket_geometry_trans(patches, t_val, g_val, thrusts[i], cx, cy, local_com_y)

        tx['angle'].set_text( f"{'BODY TILT:':>{LW}}{wrap_angle(t_val):>{VW}.2f} [°]")
        tx['omega'].set_text( f"{'ANG VEL:':>{LW}}{omegas[i]:>{VW}.2f} [°/s]")
        tx['gimbal'].set_text(f"{'GIMBAL:':>{LW}}{g_val:>{VW}.2f} [°]")
        tx['thrust'].set_text(f"{'THRUST:':>{LW}}{thrusts[i]:>{VW}.0f} [N]")
        tx['fuel'].set_text(  f"{'FUEL LEFT:':>{LW}}{masses[i]-dry_mass:>{VW}.2f} [kg]")
        tx['burned'].set_text(f"{'FUEL USED:':>{LW}}{fuel_mass-(masses[i]-dry_mass):>{VW}.2f} [kg]")
        tx['torque'].set_text(f"{'TORQUE:':>{LW}}{torques[i]:>{VW}.0f} [N·m]")
        tx['time'].set_text(  f"{'TIME:':>{LW}}{min(times[i], final_time):>{VW}.2f} [s]")
        pv1.set_text(f"   X: {cx:10.2f} m      Y: {cy:10.2f} m")
        pv2.set_text(f"  Vx: {vxs[i]:10.2f} m/s   Vy: {vys[i]:10.2f} m/s")

        show_omega_arrow = times[i] < dt and abs(omega_0) > 1e-6
        tip = get_rotated_points([(0, 50 - local_com_y)], (0,0), t_val)[0]
        tip_world = (tip[0] + cx, tip[1] + cy)

        if show_omega_arrow:
            _omega_arrow(ax, arr, tip_world[0], tip_world[1], np.sign(omega_0), arrow_len)
            vel_arr.set_visible(False)
            _update_legend(ax, [init_line, tgt_line, arr],
                           ["Initial Angle", "Target Angle", r"Initial $\omega$"])
        else:
            arr.set_visible(False)
            speed = np.sqrt(vxs[i]**2 + vys[i]**2)
            if times[i] >= final_time and speed > 1e-6:
                # Show velocity arrow after manoeuvre ends
                gc = rotate_point(0, -local_com_y, 0, 0, t_val)
                bx, by = gc[0]+cx, gc[1]+cy
                vel_arr.set_positions((bx, by),
                                      (bx + vxs[i]/speed*arrow_len, by + vys[i]/speed*arrow_len))
                vel_arr.set_visible(True)
                _update_legend(ax, [init_line, tgt_line, vel_arr],
                               ["Initial Angle", "Target Angle", "Velocity"])
            else:
                vel_arr.set_visible(False)
                _update_legend(ax, [init_line, tgt_line], ["Initial Angle", "Target Angle"])

        return (*patches, *tx.values(), pv1, pv2, arr, vel_arr)

    return fig, animate

# ===========================================================================
# Animation — Dual side-by-side mode
# ===========================================================================

def _build_anim_dual():
    max_disp       = max(max(abs(x) for x in x_coms), max(abs(y) for y in y_coms), 1)
    fig_size_rot   = rocket_height + 40
    fig_size_trans = max(1.75 * max_disp, rocket_height + 30)
    arr_len_rot    = 20
    arr_len_trans  = 0.2 * fig_size_trans

    fig = plt.figure(figsize=(19.5, 13.5))
    gs  = fig.add_gridspec(2, 2, height_ratios=[1, 0.28],
                            width_ratios=[1, 1], wspace=0.08, hspace=0.15)
    ax_r  = fig.add_subplot(gs[0, 0])
    ax_t  = fig.add_subplot(gs[0, 1])
    ax_tl = fig.add_subplot(gs[1, :])

    # Rotation view
    ax_r.set_xlim(-fig_size_rot, fig_size_rot);  ax_r.set_ylim(-fig_size_rot, fig_size_rot)
    ax_r.set_aspect('equal');  ax_r.axis('off')
    ax_r.set_title("Rotation-Only View", fontsize=20, pad=10)
    _sky_bg(ax_r, [-fig_size_rot, fig_size_rot, -fig_size_rot, fig_size_rot])
    patches_r = _add_rocket_patches(ax_r)
    rad1 = np.radians(theta_0);  rad2 = np.radians(theta_target)
    il_r, = ax_r.plot([0, 80*np.sin(rad1)], [0, 80*np.cos(rad1)], color='red',   ls='--', alpha=0.6, zorder=5)
    tl_r, = ax_r.plot([0, 80*np.sin(rad2)], [0, 80*np.cos(rad2)], color='green', ls='--', alpha=0.7, zorder=5)
    arr_r = FancyArrowPatch((0,0),(0,0), mutation_scale=20, color='yellow', visible=False)
    ax_r.add_patch(arr_r)

    # Translation view
    ax_t.set_xlim(-fig_size_trans, fig_size_trans);  ax_t.set_ylim(-fig_size_trans+25, fig_size_trans)
    ax_t.set_aspect('equal');  ax_t.axis('off')
    ax_t.set_title("Full View (Rotation + Translation)", fontsize=20, pad=10)
    _sky_bg(ax_t, [-fig_size_trans, fig_size_trans, -fig_size_trans, fig_size_trans])
    patches_t = _add_rocket_patches(ax_t)
    il_t, = ax_t.plot([0, 80*np.sin(rad1)], [0, 80*np.cos(rad1)], color='red',   ls='--', alpha=0.6, zorder=5)
    tl_t, = ax_t.plot([0, 80*np.sin(rad2)], [0, 80*np.cos(rad2)], color='green', ls='--', alpha=0.7, zorder=5)
    arr_t     = FancyArrowPatch((0,0),(0,0), mutation_scale=20, color='yellow', visible=False, zorder=20)
    vel_arr_t = FancyArrowPatch((0,0),(0,0), mutation_scale=20, color='cyan',   visible=False, zorder=20)
    ax_t.add_patch(arr_t);  ax_t.add_patch(vel_arr_t)

    # Telemetry panel
    ax_tl.set_facecolor('white');  ax_tl.axis('off')
    ax_tl.text(0.5, 0.95, "TELEMETRY", color='black', fontsize=24, fontweight='bold', ha='center')
    ax_tl.text(0.5, 1.13, rf"Initial $\omega_0$ = {omega_0:.2f} °/s", color='black', fontsize=16, ha='center')
    fs = 20
    # Labels (static)
    for x, y, label in [(0.04,0.75,"BODY TILT:"),(0.04,0.50,"ANG VEL: "),(0.04,0.25,"GIMBAL:  "),
                         (0.39,0.75,"THRUST:   "),(0.39,0.50,"FUEL LEFT:"),(0.39,0.25,"TIME:     "),
                         (0.72,0.75,"X:   "),(0.72,0.50,"Y:   "),(0.72,0.25,"VEL:")]:
        ax_tl.text(x, y, label, color='black', fontsize=fs, family='monospace', fontweight='bold')
    # Value text objects
    VW_d = 11
    tx = {k: ax_tl.text(xp, yp, '', color='black', fontsize=fs, family='monospace', fontweight='bold')
          for k, xp, yp in [('angle',0.19,0.75),('omega',0.19,0.50),('gimbal',0.19,0.25),
                              ('thrust',0.54,0.75),('fuel',0.54,0.50),('time',0.54,0.25),
                              ('x',0.82,0.75),('y',0.82,0.50),('speed',0.82,0.25)]}

    def animate(i):
        t_wrapped   = angles[i]
        t_val       = unwrapped_angles[i]
        g_val       = gimbals[i]
        cx, cy      = x_coms[i], y_coms[i]
        local_com_y = coms[i] - dry_cm_from_nozzle
        speed       = np.sqrt(vxs[i]**2 + vys[i]**2)

        _set_rocket_geometry_rot(patches_r,  t_wrapped, g_val, thrusts[i])
        _set_rocket_geometry_trans(patches_t, t_val,    g_val, thrusts[i], cx, cy, local_com_y)

        tx['angle'].set_text( f"{t_wrapped:>{VW_d}.2f} °")
        tx['omega'].set_text( f"{omegas[i]:>{VW_d}.2f} °/s")
        tx['gimbal'].set_text(f"{g_val:>{VW_d}.2f} °")
        tx['thrust'].set_text(f"{thrusts[i]:>{VW_d}.0f} N")
        tx['fuel'].set_text(  f"{masses[i]-dry_mass:>{VW_d}.2f} kg")
        tx['time'].set_text(  f"{min(times[i], final_time):>{VW_d}.2f} s")
        tx['x'].set_text(     f"{cx:>{VW_d}.2f} m")
        tx['y'].set_text(     f"{cy:>{VW_d}.2f} m")
        tx['speed'].set_text( f"{speed:>{VW_d}.2f} m/s")

        show_omega = times[i] < dt and abs(omega_0) > 1e-6

        # Rotation-view arrows
        tip_r = get_rotated_points([(0, 50)], (0,0), t_wrapped)[0]
        if show_omega:
            _omega_arrow(ax_r, arr_r, tip_r[0], tip_r[1], np.sign(omega_0), arr_len_rot)
            _update_legend(ax_r, [il_r, tl_r, arr_r],
                           ["Initial Angle", "Target Angle", r"Initial $\omega$"])
        else:
            arr_r.set_visible(False)
            _update_legend(ax_r, [il_r, tl_r], ["Initial Angle", "Target Angle"])

        # Translation-view arrows
        tip_t = get_rotated_points([(0, 50-local_com_y)], (0,0), t_val)[0]
        tip_tw = (tip_t[0]+cx, tip_t[1]+cy)
        if show_omega:
            _omega_arrow(ax_t, arr_t, tip_tw[0], tip_tw[1], np.sign(omega_0), arr_len_trans)
            vel_arr_t.set_visible(False)
            _update_legend(ax_t, [il_t, tl_t, arr_t],
                           ["Initial Angle", "Target Angle", r"Initial $\omega$"])
        else:
            arr_t.set_visible(False)
            if times[i] >= final_time and speed > 1e-6:
                gc = rotate_point(0, -local_com_y, 0, 0, t_val)
                bx, by = gc[0]+cx, gc[1]+cy
                vel_arr_t.set_positions((bx,by),
                                        (bx+vxs[i]/speed*arr_len_trans,
                                         by+vys[i]/speed*arr_len_trans))
                vel_arr_t.set_visible(True)
                _update_legend(ax_t, [il_t, tl_t, vel_arr_t],
                               ["Initial Angle", "Target Angle", "Velocity"])
            else:
                vel_arr_t.set_visible(False)
                _update_legend(ax_t, [il_t, tl_t], ["Initial Angle", "Target Angle"])

        return (*patches_r, *patches_t, *tx.values(), arr_r, arr_t, vel_arr_t)

    return fig, animate

# ===========================================================================
# Build and save animation
# ===========================================================================

if render_both:
    fig, animate = _build_anim_dual()
    out_path, fps, interval, bitrate = "../results/attitude_dual.mp4", 50, 25, 2800
elif include_translation:
    fig, animate = _build_anim_translation()
    out_path, fps, interval, bitrate = "../results/attitude_translation.mp4", 50, 20, 2500
else:
    fig, animate = _build_anim_rotation()
    out_path, fps, interval, bitrate = "../results/attitude_rotation.mp4", 50, 20, 2500

ani = FuncAnimation(fig, animate, frames=ANIM_FRAMES, interval=interval, blit=True)
print(f"Saving animation to {out_path} ...")
ani.save(out_path, writer=FFMpegWriter(fps=fps, bitrate=bitrate))
print("Done.")
plt.close()


# Full 2D Case

## SLSQP Implementation

In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Polygon, FancyArrowPatch, Rectangle
from matplotlib.colors import LinearSegmentedColormap


# ==========================================
# CENTRAL CONFIGURATION — ALL TUNABLE PARAMETERS
# All magic numbers, weights, bounds, geometry, tolerances, animation settings,
# solver options, and scenario initial conditions are gathered here.
# Modify only this section to tune behavior / difficulty / visuals.
# ==========================================
class RocketConfig:

    # 1. Physics & Propulsion
    g = 9.81                  # Gravity acceleration (m/s^2)
    I_sp = 300.0              # Specific impulse (seconds)
    g0 = 9.81                 # Standard gravity for Isp (m/s^2)
    v_e = I_sp * g0           # Exhaust velocity (m/s) computed automatically

    # 2. Mass & Thrust Limits
    m_dry = 1000.0            # Dry mass (kg)
    m_fuel = 500.0            # Initial fuel mass (kg)
    T_max = 18000.0           # Maximum engine thrust (N)
    T_min = 0.0               # Minimum throttleable thrust (N)

    # 3. Geometry (for CoM shift and moment of inertia)
    rocket_height = 95.0                # Total length nozzle → nose tip (m)
    fuel_tank_height = rocket_height / 2
    dry_cm_from_nozzle = rocket_height / 2
    fuel_cm_from_nozzle = fuel_tank_height / 2

    # 4. Control Limits
    alpha_max_deg = 10.0
    alpha_max = np.deg2rad(alpha_max_deg)   # Maximum engine gimbal angle (radians)

    # 5. Discretization
    N = 40                    # Number of time intervals / collocation points

    # 6. Optimization Settings & Bounds
    tf_min = 1.0
    tf_max = 60.0
    tf_initial_guess = 15.0

    vx_bound = 50.0           # |vx| <= this value (m/s)
    vy_upper = 50.0
    vy_lower = -100.0
    omega_bound = 0.5         # |omega| <= this value (rad/s)

    # Tight landing constraints applied to last k nodes
    landing_nodes = 5                  # Number of final nodes with tight attitude/gimbal
    tight_theta_deg = 2.0
    tight_theta = np.deg2rad(tight_theta_deg)     # +-2° near touchdown
    tight_alpha_deg = 2.0
    tight_alpha = np.deg2rad(tight_alpha_deg)     # +-2° gimbal near touchdown

    theta_loose_max = np.pi / 4        # Loose attitude bound during most of flight (+-45°)

    # 7. Objective Function Weights
    w_time = 10.0             # Primary: minimize final time
    w_thrust = 1e-3           # Thrust magnitude penalty (smooths bang-bang)
    w_gimbal = 0.05           # Gimbal angle penalty (discourages large/deflected gimbal)
    w_gimbal_rate = 0.25      # Gimbal slew rate penalty (prevents rapid oscillations)
    w_theta = 0.5             # Attitude deviation penalty (favors upright flight)
    w_alt_thrust = 5e-8       # Extra thrust penalty at high altitude

    # Ground proximity / violation penalties
    clearance_zone_factor = 1.5
    clearance_zone = rocket_height * clearance_zone_factor   # Distance over which proximity ramps up
    w_landing = 2000.0        # Very strong penalty on attitude/gimbal near ground
    ground_violation_weight = 1e6

    # 8. Animation & Visualization Parameters
    fps = 30
    hold_time_start = 2.0
    hold_time_end = 2.0

    base_flame_length = 25.0          # Flame length at full thrust (m)
    visual_margin_factor = 1.5        # Plot margin around rocket trajectory
    arrow_length_factor = 0.1         # Initial velocity arrow size relative to plot width

    # Rocket geometry (local coordinates - nozzle near y=0)
    body_raw = [[-7, 10], [7, 10], [7, 80], [-7, 80]]
    nose_raw = [[-7, 80], [7, 80], [0, 95]]
    engine_attach = (0, 10)
    engine_raw = [[-8.4, 0], [8.4, 0], [4.5, 10], [-4.5, 10]]
    flame_raw_base = [[-5.6, 0], [5.6, 0], [0, -base_flame_length]]

    # Telemetry text layout
    label_width = len("FUEL BURNED:") + 1
    value_width = 11

    # 9. Diagnostic Plot Settings
    plot_colors = ['cyan', 'lime', 'orange', 'red', 'blue', 'purple',
                   'magenta', 'gold', 'teal', 'navy', 'brown', 'olive']

    # 10. Scenario — Initial Conditions
    initial_nozzle_altitude = 230.42   # Nozzle height above ground at t=0 (m)
    initial_downrange = 20.0          # Initial horizontal offset (m)
    initial_vx = -5.0                   # Initial horizontal velocity (m/s)
    initial_vy = -15.0                 # Initial vertical velocity (m/s) - negative = downward


# Create global config instance
config = RocketConfig()
output_file_name = "SLSQP"

# ==========================================
# HELPER — CoM & Moment of Inertia
# ==========================================
def calculate_com_and_I(curr_mass):
    """Compute center of mass (from nozzle) and pitch moment of inertia for current mass."""
    fuel_remaining = max(curr_mass - config.m_dry, 0.0)
    com_from_nozzle = (config.m_dry * config.dry_cm_from_nozzle +
                       fuel_remaining * config.fuel_cm_from_nozzle) / curr_mass

    I_dry_cm = (1 / 12) * config.m_dry * config.rocket_height ** 2
    dry_distance = config.dry_cm_from_nozzle - com_from_nozzle
    I_dry = I_dry_cm + config.m_dry * dry_distance ** 2

    I_fuel_cm = (1 / 12) * fuel_remaining * config.fuel_tank_height ** 2 if fuel_remaining > 0 else 0
    fuel_distance = config.fuel_cm_from_nozzle - com_from_nozzle
    I_fuel = I_fuel_cm + fuel_remaining * fuel_distance ** 2

    I_z = I_dry + I_fuel
    return com_from_nozzle, I_z


# ==========================================
# DYNAMICS — continuous-time model
# ==========================================
def rocket_dynamics(state, control):
    """
    Continuous-time dynamics.
    State:  [x, y, vx, vy, theta, omega, m]   — y is CoM altitude
    Control:[T,   alpha]

    Convention:
      theta = 0 → rocket vertical (nose up)
      theta > 0 → tilted clockwise (to the right)
      alpha > 0 → gimbal deflects thrust clockwise relative to body
    """
    x, y, vx, vy, theta, omega, m = state
    T, alpha = control

    l_t, I_z = calculate_com_and_I(m)

    thrust_angle = theta + alpha

    dx = vx
    dy = vy
    dvx = (T / m) * np.sin(thrust_angle)
    dvy = (T / m) * np.cos(thrust_angle) - config.g
    dtheta = omega
    domega = -(l_t * T / I_z) * np.sin(alpha) if I_z > 0 else 0.0
    dm = -T / config.v_e

    return np.array([dx, dy, dvx, dvy, dtheta, domega, dm])


# ==========================================
# OPTIMIZATION PROBLEM - using scipy.minimize + SLSQP
# ==========================================
def solve_optimal_landing(initial_state):
    N = config.N
    n_states = 7
    n_controls = 2

    # Total optimization variables: states + controls + tf
    n_vars = (N + 1) * (n_states + n_controls) + 1

    # Initial guess setup
    time_array = np.linspace(0, config.tf_initial_guess, N + 1)

    # Approximate final CoM (slightly above dry mass for safety)
    approx_final_com, _ = calculate_com_and_I(config.m_dry + config.m_fuel * 0.1)
    final_state_target = np.array([0, approx_final_com, 0, 0, 0, 0, config.m_dry])

    # Linear interpolation guess for states
    x_guess = np.linspace(initial_state, final_state_target, N + 1)

    # Control guess: constant hover thrust, zero gimbal
    u_guess = np.zeros((N + 1, n_controls))
    hover_thrust = initial_state[6] * config.g
    hover_thrust = np.clip(hover_thrust, config.T_min, config.T_max)
    u_guess[:, 0] = hover_thrust
    u_guess[:, 1] = 0.0

    # Flatten into single vector: [states_flat, controls_flat, tf]
    x0 = np.concatenate([x_guess.flatten(), u_guess.flatten(), [config.tf_initial_guess]])

    def unpack(vector):
        """Unpack flat optimization vector into states, controls, tf, dt."""
        tf = vector[-1]
        dt = tf / N
        idx_state_end = (N + 1) * n_states
        states_flat = vector[:idx_state_end]
        controls_flat = vector[idx_state_end:-1]
        states = states_flat.reshape((N + 1, n_states))
        controls = controls_flat.reshape((N + 1, n_controls))
        return states, controls, tf, dt

    # ────────────────────────────────────────────────
    # OBJECTIVE FUNCTION
    # ────────────────────────────────────────────────
    def objective(vector):
        states, controls, tf, dt = unpack(vector)

        thrusts = controls[:, 0]
        gimbals = controls[:, 1]
        thetas = states[:, 4]
        ys = states[:, 1]               # CoM altitude
        masses = states[:, 6]

        # Nozzle height for ground penalties
        nozzle_heights = ys - np.array([calculate_com_and_I(m)[0] for m in masses])

        # Ground proximity (0 = high up, ~1 = near ground)
        ground_proximity = np.maximum(0, 1.0 - nozzle_heights / config.clearance_zone)
        ground_proximity = ground_proximity ** 3
        ground_violation = np.maximum(0, -nozzle_heights)

        # Costs
        J_time = config.w_time * tf
        J_thrust = config.w_thrust * dt * np.sum(thrusts ** 2)
        J_gimbal = config.w_gimbal * dt * np.sum(gimbals ** 2)

        gimbal_rates = np.diff(gimbals) / dt
        J_gimbal_rate = config.w_gimbal_rate * dt * np.sum(gimbal_rates ** 2)

        J_theta = config.w_theta * dt * np.sum(thetas ** 2)

        J_ground = config.ground_violation_weight * np.sum(ground_violation ** 2)

        J_landing_theta = config.w_landing * dt * np.sum((thetas * ground_proximity) ** 2)
        J_landing_gimbal = config.w_landing * dt * np.sum((gimbals * ground_proximity) ** 2)

        # Encourage early thrust cutoff
        y_norm = ys / np.max(ys) if np.max(ys) > 0 else np.ones_like(ys)
        J_alt_thrust = config.w_alt_thrust * dt * np.sum(thrusts ** 2 * y_norm)

        total = (J_time + J_thrust + J_gimbal + J_gimbal_rate + J_theta +
                 J_alt_thrust + J_ground + J_landing_theta + J_landing_gimbal)

        return total

    # ────────────────────────────────────────────────
    # EQUALITY CONSTRAINTS (dynamics + boundaries)
    # ────────────────────────────────────────────────
    def constraints(vector):
        states, controls, tf, dt = unpack(vector)
        cons = []

        # Trapezoidal collocation defects
        for k in range(N):
            s_k = states[k]
            s_kp1 = states[k + 1]
            u_k = controls[k]
            u_kp1 = controls[k + 1]

            f_k = rocket_dynamics(s_k, u_k)
            f_kp1 = rocket_dynamics(s_kp1, u_kp1)

            defect = s_kp1 - s_k - 0.5 * dt * (f_k + f_kp1)
            cons.extend(defect)

        # Initial condition
        cons.extend(states[0] - initial_state)

        # Terminal conditions
        cons.extend(states[-1, [0, 2, 3, 4, 5]] - [0, 0, 0, 0, 0])

        com_term, _ = calculate_com_and_I(states[-1, 6])
        cons.append(states[-1, 1] - com_term)

        # Gimbal centered at touchdown
        cons.append(controls[-1, 1])

        return np.array(cons)

    # ────────────────────────────────────────────────
    # BOUNDS
    # ────────────────────────────────────────────────
    bounds = []

    # States (N+1 nodes)
    for k in range(N + 1):
        bounds.append((None, None))                    # x
        bounds.append((0.0, None))                     # y >= 0
        bounds.append((-config.vx_bound, config.vx_bound))
        bounds.append((config.vy_lower, config.vy_upper))

        # Attitude: tighter near landing
        if k >= N + 1 - config.landing_nodes:
            bounds.append((-config.tight_theta, config.tight_theta))
        else:
            bounds.append((-config.theta_loose_max, config.theta_loose_max))

        bounds.append((-config.omega_bound, config.omega_bound))   # omega
        bounds.append((config.m_dry, initial_state[6]))            # mass

    # Controls (N+1 nodes)
    for k in range(N + 1):
        bounds.append((config.T_min, config.T_max))

        # Gimbal: tighter near landing
        if k >= N + 1 - config.landing_nodes:
            bounds.append((-config.tight_alpha, config.tight_alpha))
        else:
            bounds.append((-config.alpha_max, config.alpha_max))

    # Final time
    bounds.append((config.tf_min, config.tf_max))

    # ────────────────────────────────────────────────
    # SOLVE - two-pass strategy for robustness
    # ────────────────────────────────────────────────
    print("Optimizing trajectory (pass 1 — loose tolerance)...")
    res1 = minimize(
        objective,
        x0,
        method='SLSQP',
        constraints={'type': 'eq', 'fun': constraints},
        bounds=bounds,
        options={'maxiter': 300, 'ftol': 1e-3, 'disp': True}
    )

    x0_warm = res1.x if res1.success else x0

    print("\nOptimizing trajectory (pass 2 — tight tolerance)...")
    res = minimize(
        objective,
        x0_warm,
        method='SLSQP',
        constraints={'type': 'eq', 'fun': constraints},
        bounds=bounds,
        options={'maxiter': 500, 'ftol': 1e-6, 'disp': True}
    )

    # Use best result (prefer converged, fallback to first pass)
    best = res if (res.success or not res1.success) else res1

    if best.success or True:  # Show even approximate solutions
        states_sol, controls_sol, tf_sol, dt_sol = unpack(best.x)

        final_pos_err = np.linalg.norm(states_sol[-1, 0:2])
        final_vel_err = np.linalg.norm(states_sol[-1, 2:4])

        print(f"\n--- Solution Report ---")
        print(f"  Converged:        {best.success}")
        print(f"  Landing time:     {tf_sol:.3f} s")
        print(f"  Final pos error:  {final_pos_err:.4f} m")
        print(f"  Final vel error:  {final_vel_err:.4f} m/s")
        print(f"  Fuel consumed:    {initial_state[6] - states_sol[-1, 6]:.2f} kg")

        if not best.success:
            print(f"\n  WARNING: Optimizer did not fully converge.")
            print(f"  Message: {best.message}")
            print(f"  Solution may be approximate. Try tuning weights or initial conditions.")

        return states_sol, controls_sol, tf_sol, best.x
    else:
        print("Optimization Failed.")
        print(best.message)
        return None, None, None, None


# ==========================================
# ANIMATION & VISUALIZATION
# ==========================================
def rotate_point(x, y, pivot_x, pivot_y, angle_deg):
    angle_rad = np.radians(angle_deg)
    nx, ny = x - pivot_x, y - pivot_y
    s, c = np.sin(-angle_rad), np.cos(-angle_rad)
    rx = nx * c - ny * s
    ry = nx * s + ny * c
    return rx + pivot_x, ry + pivot_y


def get_rotated_points(points_list, pivot, angle):
    return [rotate_point(p[0], p[1], pivot[0], pivot[1], angle) for p in points_list]


def animate_results(states, controls, tf):
    N = states.shape[0] - 1
    sim_times = np.linspace(0, tf, N + 1)

    fps = config.fps
    hold_time_start = config.hold_time_start
    hold_time_end = config.hold_time_end
    num_hold_start = int(hold_time_start * fps)
    num_hold_end = int(hold_time_end * fps)
    num_anim_frames = int(tf * fps) + 1
    anim_times = np.linspace(0, tf, num_anim_frames)

    dt_anim = anim_times[1] - anim_times[0] if num_anim_frames > 1 else 0

    x_coms = interp1d(sim_times, states[:, 0])(anim_times)
    y_coms = interp1d(sim_times, states[:, 1])(anim_times)
    vxs = interp1d(sim_times, states[:, 2])(anim_times)
    vys = interp1d(sim_times, states[:, 3])(anim_times)
    angles = np.rad2deg(interp1d(sim_times, states[:, 4])(anim_times))
    omegas = np.rad2deg(interp1d(sim_times, states[:, 5])(anim_times))
    masses = interp1d(sim_times, states[:, 6])(anim_times)

    thrusts = interp1d(sim_times, controls[:, 0])(anim_times)
    gimbals = np.rad2deg(interp1d(sim_times, controls[:, 1])(anim_times))

    thrusts[-1] = 0
    gimbals[-1] = 0

    coms = np.array([calculate_com_and_I(m)[0] for m in masses])
    I_zs = np.array([calculate_com_and_I(m)[1] for m in masses])

    # Add hold periods
    x_coms = np.concatenate((np.full(num_hold_start, x_coms[0]), x_coms))
    y_coms = np.concatenate((np.full(num_hold_start, y_coms[0]), y_coms))
    vxs = np.concatenate((np.full(num_hold_start, vxs[0]), vxs))
    vys = np.concatenate((np.full(num_hold_start, vys[0]), vys))
    angles = np.concatenate((np.full(num_hold_start, angles[0]), angles))
    omegas = np.concatenate((np.full(num_hold_start, omegas[0]), omegas))
    masses = np.concatenate((np.full(num_hold_start, masses[0]), masses))
    thrusts = np.concatenate((np.full(num_hold_start, thrusts[0]), thrusts))
    gimbals = np.concatenate((np.full(num_hold_start, gimbals[0]), gimbals))
    coms = np.concatenate((np.full(num_hold_start, coms[0]), coms))
    I_zs = np.concatenate((np.full(num_hold_start, I_zs[0]), I_zs))

    x_coms = np.append(x_coms, np.full(num_hold_end, x_coms[-1]))
    y_coms = np.append(y_coms, np.full(num_hold_end, y_coms[-1]))
    vxs = np.append(vxs, np.full(num_hold_end, 0.0))
    vys = np.append(vys, np.full(num_hold_end, 0.0))
    angles = np.append(angles, np.full(num_hold_end, angles[-1]))
    omegas = np.append(omegas, np.full(num_hold_end, 0.0))
    masses = np.append(masses, np.full(num_hold_end, masses[-1]))
    thrusts = np.append(thrusts, np.full(num_hold_end, 0.0))
    gimbals = np.append(gimbals, np.full(num_hold_end, gimbals[-1]))
    coms = np.append(coms, np.full(num_hold_end, coms[-1]))
    I_zs = np.append(I_zs, np.full(num_hold_end, I_zs[-1]))

    anim_times_start = np.linspace(-hold_time_start, -dt_anim, num_hold_start)
    anim_times_end = np.linspace(tf + dt_anim, tf + hold_time_end, num_hold_end)
    anim_times = np.concatenate((anim_times_start, anim_times, anim_times_end))
    num_anim_frames += num_hold_start + num_hold_end

    margin = config.rocket_height * config.visual_margin_factor
    x_min = min(x_coms) - margin
    x_max = max(x_coms) + margin
    y_min = -config.rocket_height
    y_max = max(y_coms) + margin
    x_span = x_max - x_min
    y_span = y_max - y_min
    if x_span < y_span:
        extra = (y_span - x_span) / 2
        x_min -= extra
        x_max += extra
    elif y_span < x_span:
        extra = (x_span - y_span) / 2
        y_min -= extra
        y_max += extra

    fig, (ax, ax_info) = plt.subplots(1, 2, figsize=(12, 8), gridspec_kw={'width_ratios': [2, 1]})
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal')
    ax.axis('off')

    custom_cmap = LinearSegmentedColormap.from_list("sky", ["#102C57", "#000000"])
    ax.imshow(np.linspace(0, 1, 100).reshape(-1, 1), extent=[x_min, x_max, y_min, y_max],
              origin='lower', cmap=custom_cmap, aspect='auto')

    ax.axhspan(y_min, 0, color="#888888", zorder=1)
    ax.axhline(0, color="#B1B1B1", linewidth=2, zorder=2)

    initial_x_line, = ax.plot([x_coms[0], x_coms[0]], [0, y_max], color='red', ls='--', lw=2,
                              zorder=1.5, alpha=0.4, label="Initial x")
    target_x_line, = ax.plot([0, 0], [0, y_max], color='green', ls='--', lw=2,
                             zorder=1.5, alpha=0.4, label="Target x")

    pad = Rectangle((-20, -5), 40, 5, fc="gray", ec="black", zorder=3)
    ax.add_patch(pad)

    body_patch = Polygon([[0, 0]], fc="white", ec="black", zorder=11)
    nose_patch = Polygon([[0, 0]], fc="white", ec="black", zorder=12)
    engine_patch = Polygon([[0, 0]], fc="#D0D0D0", ec="black", zorder=10)
    flame_patch = Polygon([[0, 0]], fc="orange", ec="red", lw=1.5, zorder=9, visible=False)
    ax.add_patch(body_patch)
    ax.add_patch(nose_patch)
    ax.add_patch(engine_patch)
    ax.add_patch(flame_patch)

    arrow_length = abs(x_max - x_min) * config.arrow_length_factor
    vel_arrow = FancyArrowPatch(
        (0, 0), (0, 0),
        arrowstyle="simple, head_width=9, head_length=9, tail_width=4",
        mutation_scale=1.5, facecolor='cyan', edgecolor='black',
        linewidth=1.5, zorder=20, visible=False, label="Initial Velocity"
    )
    ax.add_patch(vel_arrow)

    pos_vel_text1 = ax.text(0.12, 0.06, '', transform=ax.transAxes, color='white',
                            fontsize=13, family='monospace', fontweight='bold', va='bottom', ha='left')
    pos_vel_text2 = ax.text(0.12, 0.02, '', transform=ax.transAxes, color='white',
                            fontsize=13, family='monospace', fontweight='bold', va='bottom', ha='left')

    ax_info.set_facecolor('white')
    ax_info.axis('off')
    left_bound = -0.2
    fontsize = 15

    txt_alt = ax_info.text(left_bound, 0.8, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
    txt_downrange = ax_info.text(left_bound, 0.7, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
    txt_theta = ax_info.text(left_bound, 0.6, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
    txt_omega = ax_info.text(left_bound, 0.5, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
    txt_gimbal = ax_info.text(left_bound, 0.4, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
    txt_thrust = ax_info.text(left_bound, 0.3, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
    txt_fuel = ax_info.text(left_bound, 0.2, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
    txt_time = ax_info.text(left_bound, 0.1, '', color='black', fontsize=fontsize, family='monospace', fontweight='bold')
    ax_info.text(0.5, 0.9, "LIVE TELEMETRY", color='black', fontsize=fontsize + 4, fontweight='bold', ha="center")

    # Geometry from config
    body_raw = config.body_raw
    nose_raw = config.nose_raw
    engine_attach = config.engine_attach
    engine_raw = config.engine_raw
    flame_raw = config.flame_raw_base

    engine_rel = [(ex - engine_attach[0], ey - engine_attach[1]) for ex, ey in engine_raw]
    flame_rel = [(fx - engine_attach[0], fy - engine_attach[1]) for fx, fy in flame_raw]

    LABEL_WIDTH = config.label_width
    VALUE_WIDTH = config.value_width

    def animate(i):
        t_val = angles[i]
        g_val = gimbals[i]
        thr_val = thrusts[i]
        cx = x_coms[i]
        cy = y_coms[i]
        vx_i, vy_i = vxs[i], vys[i]
        com = coms[i]

        body_local = [[px, py - com] for px, py in body_raw]
        nose_local = [[px, py - com] for px, py in nose_raw]
        engine_attach_local = (0, 10 - com)

        body_rot = get_rotated_points(body_local, (0, 0), t_val)
        nose_rot = get_rotated_points(nose_local, (0, 0), t_val)

        engine_gimb_rel = get_rotated_points(engine_rel, (0, 0), g_val)
        engine_local = [(ex + engine_attach_local[0], ey + engine_attach_local[1]) for ex, ey in engine_gimb_rel]
        engine_rot = get_rotated_points(engine_local, (0, 0), t_val)

        flame_length = config.base_flame_length * (thr_val / config.T_max) if config.T_max > 0 else 0
        flame_rel_scaled = [(fx, fy * (flame_length / config.base_flame_length)) for fx, fy in flame_rel]
        flame_gimb_rel = get_rotated_points(flame_rel_scaled, (0, 0), g_val)
        flame_local = [(fx + engine_attach_local[0], fy + engine_attach_local[1]) for fx, fy in flame_gimb_rel]
        flame_rot = get_rotated_points(flame_local, (0, 0), t_val)

        body_patch.set_xy([(px + cx, py + cy) for px, py in body_rot])
        nose_patch.set_xy([(px + cx, py + cy) for px, py in nose_rot])
        engine_patch.set_xy([(px + cx, py + cy) for px, py in engine_rot])
        flame_patch.set_xy([(px + cx, py + cy) for px, py in flame_rot])
        flame_patch.set_visible(thr_val > 0)

        txt_alt.set_text(f"{'ALTITUDE:':>{LABEL_WIDTH}}{y_coms[i] - coms[i]:>{VALUE_WIDTH}.2f} m")
        txt_downrange.set_text(f"{'DOWNRANGE:':>{LABEL_WIDTH}}{x_coms[i]:>{VALUE_WIDTH}.2f} m")
        txt_theta.set_text(f"{'THETA:':>{LABEL_WIDTH}}{t_val:>{VALUE_WIDTH}.2f} °")
        txt_omega.set_text(f"{'OMEGA:':>{LABEL_WIDTH}}{omegas[i]:>{VALUE_WIDTH}.2f} °/s")
        txt_gimbal.set_text(f"{'GIMBAL:':>{LABEL_WIDTH}}{g_val:>{VALUE_WIDTH}.2f} °")
        txt_thrust.set_text(f"{'THRUST:':>{LABEL_WIDTH}}{thr_val:>{VALUE_WIDTH}.0f} N")
        txt_fuel.set_text(f"{'FUEL:':>{LABEL_WIDTH}}{masses[i] - config.m_dry:>{VALUE_WIDTH}.2f} kg")
        txt_time.set_text(f"{'TIME:':>{LABEL_WIDTH}}{max(0, min(anim_times[i], tf)):>{VALUE_WIDTH}.2f} s")

        pos_vel_text1.set_text(f" X: {x_coms[i]:10.2f} m   Alt: {y_coms[i] - coms[i]:10.2f} m")
        pos_vel_text2.set_text(f"Vx: {vxs[i]:10.2f} m/s  Vy: {vys[i]:10.2f} m/s")

        show_vel_arrow = anim_times[i] < 0
        speed = np.sqrt(vx_i ** 2 + vy_i ** 2)
        if show_vel_arrow and speed > 1e-3:
            base_x, base_y = cx, cy
            dx = (vx_i / speed) * arrow_length
            dy = (vy_i / speed) * arrow_length
            vel_arrow.set_positions((base_x, base_y), (base_x + dx, base_y + dy))
            vel_arrow.set_visible(True)
        else:
            vel_arrow.set_visible(False)

        legend = ax.get_legend()
        if legend is not None:
            legend.remove()

        if show_vel_arrow and speed > 1e-3:
            handles = [initial_x_line, target_x_line, vel_arrow]
            labels = ["Initial x", "Target x", "Initial Velocity"]
            ax.legend(handles, labels, loc='upper right', fontsize=10, framealpha=0.7)
        else:
            handles = [initial_x_line, target_x_line]
            labels = ["Initial x", "Target x"]
            ax.legend(handles, labels, loc='upper right', fontsize=10, framealpha=0.7)

        return (body_patch, nose_patch, engine_patch, flame_patch,
                txt_alt, txt_downrange, txt_theta, txt_omega, txt_gimbal,
                txt_thrust, txt_fuel, txt_time, pos_vel_text1, pos_vel_text2, vel_arrow)

    ani = FuncAnimation(fig, animate, frames=range(num_anim_frames), interval=1000 // fps, blit=True)

    print("Saving animation...")
    writer = FFMpegWriter(fps=fps, bitrate=2500)
    ani.save(f"../results/{output_file_name}.mp4", writer=writer)
    print("Done.")
    plt.show()


# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    print("--- Time-Optimal Rocket Landing Solver (scipy SLSQP) ---")

    # Initial conditions from central config
    h_0 = config.initial_nozzle_altitude
    x_0 = config.initial_downrange
    vx_0 = config.initial_vx
    vy_0 = config.initial_vy

    initial_m = config.m_dry + config.m_fuel
    initial_com, _ = calculate_com_and_I(initial_m)

    s0 = np.array([x_0, h_0 + initial_com, vx_0, vy_0, 0.0, 0.0, initial_m])

    print(f"\nInitial state: {s0}")

    states, controls, tf, raw_res = solve_optimal_landing(s0)

    if states is not None:
        # Diagnostic plots
        fig_diag, axs = plt.subplots(12, 1, figsize=(10, 26), sharex=True)
        plt.subplots_adjust(hspace=0.28)

        times = np.linspace(0, tf, states.shape[0])
        angles_deg = np.rad2deg(states[:, 4])
        omegas_deg = np.rad2deg(states[:, 5])
        gimbals_deg = np.rad2deg(controls[:, 1])
        masses = states[:, 6]
        thrusts = controls[:, 0]
        coms = np.array([calculate_com_and_I(m)[0] for m in masses])
        I_zs = np.array([calculate_com_and_I(m)[1] for m in masses])
        torques = coms * thrusts * np.sin(np.radians(gimbals_deg))
        x_coms = states[:, 0]
        y_coms = states[:, 1]
        vxs = states[:, 2]
        vys = states[:, 3]

        plot_data = [angles_deg, omegas_deg, gimbals_deg, masses, thrusts, torques,
                     coms, I_zs, x_coms, y_coms, vxs, vys]
        labels = ['Angle (deg)', 'Ω (deg/s)', 'Gimbal (deg)', 'Mass (kg)', 'Thrust (N)',
                  'Torque (N·m)', 'CoM from thrust (m)', 'Moment of Inertia (kg·m²)',
                  'X pos (m)', 'Y pos (m)', 'Vx (m/s)', 'Vy (m/s)']

        for i, ax in enumerate(axs):
            ax.plot(times, plot_data[i], color=config.plot_colors[i])
            ax.set_ylabel(labels[i])
            ax.set_facecolor("#ffffff")
            ax.grid(True, alpha=0.18)
            ax.set_xlim(-0.4, times[-1] + 0.6)

        axs[0].axhline(0, color='black', linestyle='--', alpha=0.6, label="target")
        axs[0].legend(fontsize=9)
        axs[7].axhline(I_zs[0], color='darkgreen', ls='--', alpha=0.5, label=f"I = {I_zs[0]:.0f}")
        axs[7].legend(fontsize=9, loc='upper right')
        axs[-1].set_xlabel("Time (s)")

        plt.suptitle("Post-Flight Analysis", fontsize=16, y=0.995)
        plt.show()

        # Generate animation
        animate_results(states, controls, tf)

## CasADi Implementation

In [ ]:
import numpy as np
import casadi as ca
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Polygon, FancyArrowPatch, Rectangle
from matplotlib.colors import LinearSegmentedColormap
from scipy.interpolate import interp1d


# ===========================================================================
# Configuration
# All tunable parameters live here. Modify this section to change the scenario.
# ===========================================================================
class RocketConfig:

    # Physics
    g    = 9.81           # gravity (m/s^2)
    I_sp = 500.0          # specific impulse (s)
    g0   = 9.81           # standard gravity for Isp (m/s^2)
    v_e  = I_sp * g0      # exhaust velocity (m/s)

    # Vehicle
    m_dry  = 1250.0       # dry mass (kg)
    m_fuel = 500.0        # fuel mass (kg)
    T_max  = 50000.0      # maximum thrust (N)
    T_min  = 4000.0       # minimum throttleable thrust (N)

    rocket_height       = 95.0
    fuel_tank_height    = rocket_height / 2
    dry_cm_from_nozzle  = rocket_height / 2
    fuel_cm_from_nozzle = fuel_tank_height / 2

    # Control
    ALPHA_MAX_DEG = 10.0
    alpha_max     = np.deg2rad(ALPHA_MAX_DEG)   # gimbal limit (rad)

    # Discretisation
    N = 40                # direct-collocation nodes

    # Optimisation bounds
    tf_min = 1.0;   tf_max = 60.0
    vx_min = -50.0; vx_max = 50.0
    vy_min = -100.0; vy_max = 50.0
    omega_min = -0.5; omega_max = 0.5

    # Attitude constraints
    theta_max_loose = np.pi / 4       # loose bound during flight
    LANDING_NODES   = 6               # final nodes with tight constraints
    tight_theta     = np.deg2rad(2.0)
    tight_alpha     = np.deg2rad(2.0)
    initial_tf_guess = 15.0

    # Objective weights
    epsilon       = 5.0    # nozzle-height softening (m)
    w_time        = 10.0
    w_thrust      = 1e-10
    w_gimbal      = 0.05
    w_gimbal_rate = 0.8
    w_theta       = 80.0
    w_alt_thrust  = 6.25e-6
    w_landing     = 4000.0
    w_v           = 50.0   # velocity penalty near ground

    # Ground proximity penalty
    clearance_zone_factor = 2.0
    clearance_zone        = rocket_height * clearance_zone_factor

    # Pre-flight feasibility thresholds
    nozzle_ground_threshold    = -0.1
    hover_thrust_safety_factor = 0.6
    dv_available_margin        = 0.6
    initial_theta_max_deg      = 25
    initial_omega_max_deg      = 35
    initial_x_max              = 800
    dv_gravity_loss_factor     = 1.5
    dv_time_margin             = 5.0

    # Animation
    base_flame_length          = 35.0
    fps                        = 30
    hold_time_start            = 2.0
    hold_time_end              = 2.0
    flame_throttle_gamma       = 0.5   # <1 --> low-thrust flames look stronger
    visual_margin_factor       = 1.5
    visual_arrow_length_factor = 0.1

    # Drawing geometry (nozzle base near y=0)
    body_raw      = [[-7, 10], [7, 10], [7, 80], [-7, 80]]
    nose_raw      = [[-7, 80], [7, 80], [0, 95]]
    engine_attach = (0, 10)
    engine_raw    = [[-8.4, 0], [8.4, 0], [4.5, 10], [-4.5, 10]]
    flame_raw     = [[-5.6, 0], [5.6, 0], [0, -base_flame_length]]
    label_width   = len("FUEL BURNED:") + 1
    value_width   = 11

    # Landing quality tolerances
    landing_pos_tol      = 0.10
    landing_vel_tol      = 0.15
    min_nozzle_tol       = -0.30
    skidding_x_threshold = 5.0

    # Initial conditions
    initial_nozzle_altitude = 160.42
    initial_x_0  =  30.0
    initial_vx_0 =  -8.0
    initial_vy_0 = -30.0


config          = RocketConfig()
output_file_name = "new_casadi_test"


# ===========================================================================
# NumPy mass-properties helper (used in plots and animation)
# ===========================================================================

def calculate_com_and_I(curr_mass):
    """Return (CoM from nozzle [m], moment of inertia [kg·m²]) for curr_mass."""
    fuel = max(curr_mass - config.m_dry, 0.0)
    com  = (config.m_dry * config.dry_cm_from_nozzle +
            fuel        * config.fuel_cm_from_nozzle) / curr_mass

    I_dry  = ((1/12)*config.m_dry*config.rocket_height**2
              + config.m_dry*(config.dry_cm_from_nozzle - com)**2)
    I_fuel = ((1/12)*fuel*config.fuel_tank_height**2
              + fuel*(config.fuel_cm_from_nozzle - com)**2) if fuel > 0 else 0.0
    return com, I_dry + I_fuel


# ===========================================================================
# Pre-flight feasibility check
# ===========================================================================

def approx_controllable(s0, config):
    """
    Rough sanity check: is the landing physically plausible?
    Returns True if the scenario looks feasible.
    """
    x, y, vx, vy, theta, omega, m = s0
    fuel = m - config.m_dry

    if fuel < 1e-6:
        print("No usable fuel.")
        return False

    com, _ = calculate_com_and_I(m)
    if (y - com) < config.nozzle_ground_threshold:
        return False

    if config.T_max < config.hover_thrust_safety_factor * m * config.g:
        print("Thrust too low to hover.")
        return False

    dv_available = config.v_e * np.log(m / config.m_dry)
    speed        = np.sqrt(vx**2 + vy**2)
    dv_required  = speed + config.dv_gravity_loss_factor * config.g * (
                       abs(vy)/config.g + config.dv_time_margin)
    if dv_available < config.dv_available_margin * dv_required:
        print(f"Δv budget too low: {dv_available:.1f} <~ {dv_required:.1f} m/s")
        return False

    if (abs(theta) > np.deg2rad(config.initial_theta_max_deg) or
            abs(omega) > np.deg2rad(config.initial_omega_max_deg)):
        print("Initial attitude or rate too extreme.")
        return False

    if abs(x) > config.initial_x_max:
        print("Too far downrange.")
        return False

    return True


# ===========================================================================
# CasADi optimal landing solver
# ===========================================================================

def solve_optimal_landing(s0):
    """
    Formulate and solve the time-optimal landing NLP with CasADi + IPOPT.

    State  X: [x, y, vx, vy, θ, ω, m]   (7 × N+1)
    Control U: [T, α]                     (2 × N+1)

    Returns (states [N+1,7], controls [N+1,2], tf, success).
    """
    print("Setting up CasADi + IPOPT NLP...")
    opti = ca.Opti()
    N    = config.N

    # Decision variables
    X  = opti.variable(7, N+1)   # states
    U  = opti.variable(2, N+1)   # controls
    tf = opti.variable()          # free final time
    dt = tf / N

    xs, ys, vxs, vys, thetas, omegas, ms = (X[r, :] for r in range(7))
    Ts, alphas = U[0, :], U[1, :]

    # --- CasADi mass-properties (symbolic) ---
    def _com_I(m):
        fuel = m - config.m_dry
        com  = (config.m_dry * config.dry_cm_from_nozzle +
                fuel        * config.fuel_cm_from_nozzle) / m
        I_dry  = ((1/12)*config.m_dry*config.rocket_height**2
                  + config.m_dry*(config.dry_cm_from_nozzle - com)**2)
        I_fuel = ((1/12)*fuel*config.fuel_tank_height**2
                  + fuel*(config.fuel_cm_from_nozzle - com)**2)
        return com, I_dry + I_fuel

    # --- Equations of motion ---
    def _dynamics(state, control):
        x, y, vx, vy, theta, omega, m = (state[r] for r in range(7))
        T, alpha = control[0], control[1]
        l_t, I_z = _com_I(m)
        thrust_angle = theta + alpha
        return ca.vertcat(
            vx,
            vy,
            (T/m) * ca.sin(thrust_angle),
            (T/m) * ca.cos(thrust_angle) - config.g,
            omega,
            -(l_t * T / I_z) * ca.sin(alpha),
            -T / config.v_e,
        )

    # --- Trapezoidal collocation ---
    for k in range(N):
        f_k   = _dynamics(X[:, k],   U[:, k])
        f_kp1 = _dynamics(X[:, k+1], U[:, k+1])
        opti.subject_to(X[:, k+1] == X[:, k] + 0.5*dt*(f_k + f_kp1))

    # --- Boundary conditions ---
    opti.subject_to(X[:, 0] == s0)
    opti.subject_to(X[0, -1] == 0)    # x  = 0
    opti.subject_to(X[2, -1] == 0)    # vx = 0
    opti.subject_to(X[3, -1] == 0)    # vy = 0
    opti.subject_to(X[4, -1] == 0)    # θ  = 0
    opti.subject_to(X[5, -1] == 0)    # ω  = 0
    com_f, _ = _com_I(X[6, -1])
    opti.subject_to(X[1, -1] == com_f)  # y = CoM height at touchdown
    opti.subject_to(U[1, -1] == 0.0)    # gimbal centred at touchdown

    # --- Box constraints ---
    opti.subject_to(opti.bounded(config.tf_min, tf, config.tf_max))
    opti.subject_to(opti.bounded(0.0,            ys,     ca.inf))
    opti.subject_to(opti.bounded(config.vx_min,  vxs,    config.vx_max))
    opti.subject_to(opti.bounded(config.vy_min,  vys,    config.vy_max))
    opti.subject_to(opti.bounded(config.omega_min, omegas, config.omega_max))
    opti.subject_to(opti.bounded(config.m_dry,   ms,     s0[6]))
    opti.subject_to(opti.bounded(config.T_min,   Ts,     config.T_max))

    # Tight attitude constraints near landing
    for k in range(N+1):
        if k >= N+1 - config.LANDING_NODES:
            opti.subject_to(opti.bounded(-config.tight_theta, thetas[k], config.tight_theta))
            opti.subject_to(opti.bounded(-config.tight_alpha, alphas[k], config.tight_alpha))
        else:
            opti.subject_to(opti.bounded(-config.theta_max_loose, thetas[k], config.theta_max_loose))
            opti.subject_to(opti.bounded(-config.alpha_max,       alphas[k], config.alpha_max))

    # --- Objective ---
    J_time        = config.w_time        * tf
    J_thrust      = config.w_thrust      * dt * ca.sumsqr(Ts)
    J_gimbal      = config.w_gimbal      * dt * ca.sumsqr(alphas)
    J_gimbal_rate = config.w_gimbal_rate * dt * ca.sumsqr(ca.diff(alphas) / dt)
    J_theta       = config.w_theta       * dt * ca.sumsqr(thetas)

    # Ground-proximity penalties
    com_heights    = ca.horzcat(*[_com_I(ms[k])[0] for k in range(N+1)])
    nozzle_heights = ys - com_heights
    ground_prox    = ca.fmax(0, 1.0 - nozzle_heights/config.clearance_zone)**3
    h_safe         = ca.fmax(config.epsilon, nozzle_heights)

    J_inv_h        = config.w_v       * dt * ca.sum2((vxs**2 + vys**2) / h_safe)
    J_ground       = 1e8               * ca.sumsqr(ca.fmax(0, -nozzle_heights))
    J_land_theta   = config.w_landing * dt * ca.sumsqr(thetas * ground_prox)
    J_land_gimbal  = config.w_landing * dt * ca.sumsqr(alphas * ground_prox)
    J_alt_thrust   = config.w_alt_thrust * dt * ca.sum2((Ts**2) * (ys/s0[1]))

    opti.minimize(J_time + J_thrust + J_gimbal + J_gimbal_rate + J_theta
                  + J_alt_thrust + J_ground + J_land_theta + J_land_gimbal + J_inv_h)

    # --- Initial guess: linear interpolation to landed state ---
    opti.set_initial(tf, config.initial_tf_guess)
    target_com, _   = calculate_com_and_I(config.m_dry)
    target_state    = np.array([0, target_com, 0, 0, 0, 0, config.m_dry])
    for i in range(7):
        opti.set_initial(X[i, :], np.linspace(s0[i], target_state[i], N+1))
    opti.set_initial(Ts,     np.clip(s0[6]*config.g, config.T_min, config.T_max))
    opti.set_initial(alphas, 0.0)

    # --- Solve ---
    opti.solver("ipopt", {"expand": True}, {"max_iter": 500, "tol": 1e-6, "print_level": 5})
    print("\nStarting Optimisation with IPOPT...")
    try:
        sol     = opti.solve()
        success = True
        print("\n--- Solution Report ---")
        print(f"  Converged: True")
    except RuntimeError:
        sol     = opti.debug
        success = False
        print("\n  WARNING: Did not fully converge. Solution may be approximate.")

    states_sol   = sol.value(X).T
    controls_sol = sol.value(U).T
    tf_sol       = sol.value(tf)

    com_f, _      = calculate_com_and_I(states_sol[-1, 6])
    final_pos_err = np.linalg.norm([states_sol[-1, 0], states_sol[-1, 1] - com_f])
    final_vel_err = np.linalg.norm(states_sol[-1, 2:4])
    print(f"  Landing time:    {tf_sol:.3f} s")
    print(f"  Final pos error: {final_pos_err:.4f} m")
    print(f"  Final vel error: {final_vel_err:.4f} m/s")
    print(f"  Fuel consumed:   {s0[6] - states_sol[-1, 6]:.2f} kg")
    return states_sol, controls_sol, tf_sol, success


# ===========================================================================
# Animation
# ===========================================================================

def rotate_point(x, y, pivot_x, pivot_y, angle_deg):
    angle_rad = np.radians(angle_deg)
    nx, ny = x - pivot_x, y - pivot_y
    s, c   = np.sin(-angle_rad), np.cos(-angle_rad)
    return nx*c - ny*s + pivot_x, nx*s + ny*c + pivot_y

def get_rotated_points(pts, pivot, angle):
    return [rotate_point(p[0], p[1], pivot[0], pivot[1], angle) for p in pts]


def animate_results(states, controls, tf):
    N         = states.shape[0] - 1
    sim_times = np.linspace(0, tf, N+1)

    fps        = config.fps
    n_start    = int(config.hold_time_start * fps)
    n_end      = int(config.hold_time_end   * fps)
    n_frames   = int(tf * fps) + 1
    anim_times = np.linspace(0, tf, n_frames)
    dt_anim    = anim_times[1] - anim_times[0] if n_frames > 1 else 0

    # Interpolate solution onto animation time grid
    def interp(arr): return interp1d(sim_times, arr)(anim_times)
    x_coms  = interp(states[:, 0]);  y_coms  = interp(states[:, 1])
    vxs     = interp(states[:, 2]);  vys     = interp(states[:, 3])
    angles  = np.rad2deg(interp(states[:, 4]))
    omegas  = np.rad2deg(interp(states[:, 5]))
    masses  = interp(states[:, 6])
    thrusts = interp(controls[:, 0]);  thrusts[-1] = 0
    gimbals = np.rad2deg(interp(controls[:, 1]));  gimbals[-1] = 0
    coms    = np.array([calculate_com_and_I(m)[0] for m in masses])

    # Prepend start hold, append end hold
    def _pad(arr, val_start=None, val_end=0.0):
        vs = arr[0]  if val_start is None else val_start
        ve = arr[-1] if val_end    is None else val_end
        return np.concatenate((np.full(n_start, vs), arr, np.full(n_end, ve)))

    x_coms  = _pad(x_coms,  val_end=None); y_coms  = _pad(y_coms,  val_end=None)
    vxs     = _pad(vxs);                   vys     = _pad(vys)
    angles  = _pad(angles,  val_end=None); omegas  = _pad(omegas)
    masses  = _pad(masses,  val_end=None); thrusts = _pad(thrusts)
    gimbals = _pad(gimbals, val_end=None); coms    = _pad(coms, val_end=None)

    t_start    = np.linspace(-config.hold_time_start, -dt_anim, n_start)
    t_end_arr  = np.linspace(tf + dt_anim, tf + config.hold_time_end, n_end)
    anim_times = np.concatenate((t_start, anim_times, t_end_arr))
    n_frames  += n_start + n_end

    # Plot limits (square viewport)
    margin  = config.rocket_height * config.visual_margin_factor
    x_min, x_max = min(x_coms) - margin, max(x_coms) + margin
    y_min, y_max = -config.rocket_height, max(y_coms) + margin
    x_span, y_span = x_max - x_min, y_max - y_min
    if x_span < y_span:
        extra = (y_span - x_span)/2;  x_min -= extra;  x_max += extra
    elif y_span < x_span:
        extra = (x_span - y_span)/2;  y_min -= extra;  y_max += extra

    fig, (ax, ax_info) = plt.subplots(1, 2, figsize=(12, 8), gridspec_kw={'width_ratios': [2,1]})
    ax.set_xlim(x_min, x_max);  ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal');      ax.axis('off')

    sky_cmap = LinearSegmentedColormap.from_list("sky", ["#102C57", "#000000"])
    ax.imshow(np.linspace(0, 1, 100).reshape(-1, 1),
              extent=[x_min, x_max, y_min, y_max], origin='lower', cmap=sky_cmap, aspect='auto')
    ax.axhspan(y_min, 0, color="#888888", zorder=1)
    ax.axhline(0, color="#B1B1B1", linewidth=2, zorder=2)
    ax.add_patch(Rectangle((-20, -5), 40, 5, fc="gray", ec="black", zorder=3))

    ix_line, = ax.plot([x_coms[0], x_coms[0]], [0, y_max],
                       color='red',   ls='--', lw=2, zorder=1.5, alpha=0.4, label="Initial x")
    tx_line, = ax.plot([0, 0], [0, y_max],
                       color='green', ls='--', lw=2, zorder=1.5, alpha=0.4, label="Target x")

    body_p   = Polygon([[0,0]], fc="white",   ec="black", zorder=11)
    nose_p   = Polygon([[0,0]], fc="white",   ec="black", zorder=12)
    engine_p = Polygon([[0,0]], fc="#D0D0D0", ec="black", zorder=10)
    flame_p  = Polygon([[0,0]], fc="orange",  ec="red",   lw=1.5, zorder=9, visible=False)
    for p in [body_p, nose_p, engine_p, flame_p]:
        ax.add_patch(p)

    arrow_len = abs(x_max - x_min) * config.visual_arrow_length_factor
    vel_arrow = FancyArrowPatch((0,0),(0,0),
                                arrowstyle="simple, head_width=9, head_length=9, tail_width=4",
                                mutation_scale=1.5, facecolor='cyan', edgecolor='black',
                                linewidth=1.5, zorder=20, visible=False, label="Initial Velocity")
    ax.add_patch(vel_arrow)

    pv1 = ax.text(0.12, 0.06, '', transform=ax.transAxes, color='white',
                  fontsize=13, family='monospace', fontweight='bold', va='bottom', ha='left')
    pv2 = ax.text(0.12, 0.02, '', transform=ax.transAxes, color='white',
                  fontsize=13, family='monospace', fontweight='bold', va='bottom', ha='left')

    ax_info.set_facecolor('white');  ax_info.axis('off')
    lb, fs = -0.2, 15
    LW, VW = config.label_width, config.value_width
    ax_info.text(0.5, 0.9, "TELEMETRY", color='black', fontsize=fs+4, fontweight='bold', ha="center")
    tx_alt      = ax_info.text(lb, 0.8, '', color='black', fontsize=fs, family='monospace', fontweight='bold')
    tx_downrng  = ax_info.text(lb, 0.7, '', color='black', fontsize=fs, family='monospace', fontweight='bold')
    tx_theta    = ax_info.text(lb, 0.6, '', color='black', fontsize=fs, family='monospace', fontweight='bold')
    tx_omega    = ax_info.text(lb, 0.5, '', color='black', fontsize=fs, family='monospace', fontweight='bold')
    tx_gimbal   = ax_info.text(lb, 0.4, '', color='black', fontsize=fs, family='monospace', fontweight='bold')
    tx_thrust   = ax_info.text(lb, 0.3, '', color='black', fontsize=fs, family='monospace', fontweight='bold')
    tx_fuel     = ax_info.text(lb, 0.2, '', color='black', fontsize=fs, family='monospace', fontweight='bold')
    tx_time     = ax_info.text(lb, 0.1, '', color='black', fontsize=fs, family='monospace', fontweight='bold')

    # Geometry pre-computation
    eng_attach = config.engine_attach
    eng_rel    = [(ex - eng_attach[0], ey - eng_attach[1]) for ex, ey in config.engine_raw]
    flm_rel    = [(fx - eng_attach[0], fy - eng_attach[1]) for fx, fy in config.flame_raw]
    base_fl    = config.base_flame_length
    gamma      = config.flame_throttle_gamma

    def animate(i):
        t_val, g_val, thr_val = angles[i], gimbals[i], thrusts[i]
        cx, cy, com           = x_coms[i], y_coms[i], coms[i]

        # Body/nose shifted so CoM is at origin
        body_local = [[px, py - com] for px, py in config.body_raw]
        nose_local = [[px, py - com] for px, py in config.nose_raw]
        ea_local   = (0, eng_attach[1] - com)

        # Engine rotated by gimbal, then by body angle
        eng_g = get_rotated_points(eng_rel, (0,0), g_val)
        eng_l = [(ex + ea_local[0], ey + ea_local[1]) for ex, ey in eng_g]

        # Flame scaled by throttle, then rotated
        throttle    = thr_val / config.T_max if config.T_max > 0 else 0.0
        fl_len      = base_fl * (throttle ** gamma)
        flm_scaled  = [(fx, fy * fl_len / base_fl) for fx, fy in flm_rel]
        flm_g       = get_rotated_points(flm_scaled, (0,0), g_val)
        flm_l       = [(fx + ea_local[0], fy + ea_local[1]) for fx, fy in flm_g]

        def _world(pts): return [(px+cx, py+cy) for px,py in get_rotated_points(pts, (0,0), t_val)]
        body_p.set_xy(  _world(body_local)); nose_p.set_xy(  _world(nose_local))
        engine_p.set_xy(_world(eng_l));      flame_p.set_xy( _world(flm_l))
        flame_p.set_visible(thr_val > 0)

        tx_alt.set_text(    f"{'ALTITUDE:':>{LW}}{cy - com:>{VW}.2f} m")
        tx_downrng.set_text(f"{'DOWNRANGE:':>{LW}}{cx:>{VW}.2f} m")
        tx_theta.set_text(  f"{'THETA:':>{LW}}{t_val:>{VW}.2f} °")
        tx_omega.set_text(  f"{'OMEGA:':>{LW}}{omegas[i]:>{VW}.2f} °/s")
        tx_gimbal.set_text( f"{'GIMBAL:':>{LW}}{g_val:>{VW}.2f} °")
        tx_thrust.set_text( f"{'THRUST:':>{LW}}{thr_val:>{VW}.0f} N")
        tx_fuel.set_text(   f"{'FUEL:':>{LW}}{masses[i] - config.m_dry:>{VW}.2f} kg")
        tx_time.set_text(   f"{'TIME:':>{LW}}{max(0, min(anim_times[i], tf)):>{VW}.2f} s")
        pv1.set_text(f" X: {cx:10.2f} m   Alt: {cy - com:10.2f} m")
        pv2.set_text(f"Vx: {vxs[i]:10.2f} m/s  Vy: {vys[i]:10.2f} m/s")

        # Initial velocity arrow (shown during start hold)
        show_arrow = anim_times[i] < 0
        speed      = np.sqrt(vxs[i]**2 + vys[i]**2)
        if show_arrow and speed > 1e-3:
            dx = (vxs[i]/speed) * arrow_len;  dy = (vys[i]/speed) * arrow_len
            vel_arrow.set_positions((cx, cy), (cx+dx, cy+dy))
            vel_arrow.set_visible(True)
        else:
            vel_arrow.set_visible(False)

        leg = ax.get_legend()
        if leg is not None:
            leg.remove()
        if show_arrow and speed > 1e-3:
            ax.legend([ix_line, tx_line, vel_arrow], ["Initial x", "Target x", "Initial v"],
                      loc='upper right', fontsize=14, framealpha=0.7)
        else:
            ax.legend([ix_line, tx_line], ["Initial x", "Target x"],
                      loc='upper right', fontsize=14, framealpha=0.7)

        return (body_p, nose_p, engine_p, flame_p,
                tx_alt, tx_downrng, tx_theta, tx_omega,
                tx_gimbal, tx_thrust, tx_fuel, tx_time,
                pv1, pv2, vel_arrow)

    ani = FuncAnimation(fig, animate, frames=range(n_frames), interval=1000//fps, blit=True)
    print("Saving animation...")
    ani.save(f"../results/{output_file_name}.mp4", writer=FFMpegWriter(fps=fps, bitrate=2500))
    print("Done.")
    plt.show()


# ===========================================================================
# Main
# ===========================================================================

if __name__ == "__main__":
    print("--- Time-Optimal Rocket Landing Solver ---")

    h_0  = config.initial_nozzle_altitude
    x_0  = config.initial_x_0 or 0.0001   # small perturbation avoids NLP degeneracy at x=0
    vx_0 = config.initial_vx_0
    vy_0 = config.initial_vy_0

    initial_m   = config.m_dry + config.m_fuel
    initial_com, _ = calculate_com_and_I(initial_m)

    s0 = np.array([x_0, h_0 + initial_com, vx_0, vy_0, 0.0, 0.0, initial_m])

    print(f"\nInitial state: {s0}")
    print(f"  Fuel: {config.m_fuel:.1f} kg  |  Dry: {config.m_dry:.1f} kg"
          f"  |  CoM: {initial_com:.2f} m  |  Nozzle alt: {h_0:.1f} m")

    print("\nChecking approximate controllability...")
    if not approx_controllable(s0, config):
        print("--> Not controllable. Adjust m_fuel, vy_0, or altitude.")
    else:
        print("--> Check passed. Starting optimisation...")
        states, controls, tf, success = solve_optimal_landing(s0)

        if not success:
            print("\nOptimisation did NOT converge.")
        else:
            print("\nOptimisation finished. Evaluating solution quality...")

            times       = np.linspace(0, tf, states.shape[0])
            coms        = np.array([calculate_com_and_I(m)[0] for m in states[:, 6]])
            I_zs        = np.array([calculate_com_and_I(m)[1] for m in states[:, 6]])
            angles_deg  = np.rad2deg(states[:, 4])
            omegas_deg  = np.rad2deg(states[:, 5])
            gimbals_deg = np.rad2deg(controls[:, 1])
            torques     = coms * controls[:, 0] * np.sin(np.radians(gimbals_deg))
            y_nozzle    = states[:, 1] - coms

            # Diagnostic plots
            fig_diag, axs = plt.subplots(12, 1, figsize=(10, 26), sharex=True)
            plt.subplots_adjust(hspace=0.28)
            plot_data = [angles_deg, omegas_deg, gimbals_deg, states[:,6], controls[:,0], torques,
                         coms, I_zs, states[:,0], y_nozzle, states[:,2], states[:,3]]
            plot_labels = [r'$\theta$ (deg)', r'$\omega$ (deg/s)', 'Gimbal (deg)', 'Mass (kg)',
                           'Thrust (N)', 'Torque (N·m)', 'CoM from nozzle (m)', 'MoI (kg·m²)',
                           'X position (m)', 'Nozzle height (m)', 'Vx (m/s)', 'Vy (m/s)']
            plot_colors = ['cyan','lime','orange','red','blue','purple',
                           'magenta','gold','teal','navy','brown','olive']
            for i, ax in enumerate(axs):
                ax.plot(times, plot_data[i], color=plot_colors[i], lw=1.4)
                ax.set_ylabel(plot_labels[i], fontsize=11)
                ax.grid(True, alpha=0.18, ls='--');  ax.tick_params(labelsize=9)
            axs[9].axhspan(-5, 0, facecolor='gray', alpha=0.15)
            axs[9].axhline(0, color='darkred', lw=1.2, ls='--', alpha=0.7)
            plt.suptitle("Post-Flight Analysis", fontsize=16, y=0.995)
            plt.xlabel("Time (s)", fontsize=11)
            plt.show()

            # Landing quality check
            com_f, _      = calculate_com_and_I(states[-1, 6])
            error_x       = states[-1, 0]
            error_y       = states[-1, 1] - com_f
            final_pos_err = np.linalg.norm([error_x, error_y])
            final_vel_err = np.linalg.norm(states[-1, 2:4])
            min_nozzle    = float(np.min(y_nozzle))
            skidding      = any(y_nozzle[i] < 0.1 and abs(states[i,0]) > config.skidding_x_threshold
                                for i in range(len(y_nozzle)-1))

            LANDING_GOOD = (final_pos_err < config.landing_pos_tol and
                            final_vel_err < config.landing_vel_tol and
                            min_nozzle    >= config.min_nozzle_tol  and
                            not skidding)

            if LANDING_GOOD:
                print("\nLanding within tolerances!")
                print(f"  X-offset: {error_x:.3f} m  |  Y-offset: {error_y:.3f} m"
                      f"  |  Vel err: {final_vel_err:.3f} m/s  |  Min nozzle: {min_nozzle:.3f} m")
                print("Generating animation...\n")
                animate_results(states, controls, tf)
            else:
                print("\nLanding quality NOT acceptable:")
                if final_pos_err >= config.landing_pos_tol: print(f"  • pos error = {final_pos_err:.3f} m")
                if final_vel_err >= config.landing_vel_tol: print(f"  • vel error = {final_vel_err:.3f} m/s")
                if min_nozzle    <  config.min_nozzle_tol:  print(f"  • ground clip = {abs(min_nozzle):.3f} m")
                if skidding:                                 print("  • skidding detected")
                print("Animation will NOT be generated.")


# Testing (Convergence & Reachable Set)

25 Runs at N = 20: 7.2s \
25 Runs at N = 30: 14.7s \
25 Runs at N = 40: 22.6s \
25 Runs at N = 50: 29.6s \
25 Runs at N = 60: 35.6s \
25 Runs at N = 70: 51.4 s \
25 Runs at N = 80: 59.1s 

In [ ]:
import numpy as np
import casadi as ca
import matplotlib.pyplot as plt
import sys
try:
    from IPython.display import clear_output
    IN_NOTEBOOK = True
except ImportError:
    IN_NOTEBOOK = False


# ==========================================
# CENTRAL CONFIGURATION
# ==========================================
class RocketConfig:
    g = 9.81
    I_sp = 500.0
    g0 = 9.81
    v_e = I_sp * g0

    m_dry = 4000.0
    m_fuel = 150.0
    T_max = 60000.0
    T_min = 4000.0

    rocket_height = 95.0
    fuel_tank_height = rocket_height / 2
    dry_cm_from_nozzle = rocket_height / 2
    fuel_cm_from_nozzle = fuel_tank_height / 2

    ALPHA_MAX_DEG = 10.0
    alpha_max = np.deg2rad(ALPHA_MAX_DEG)

    N = 30

    tf_min = 1.0
    tf_max = 60.0

    vx_min = -50.0
    vx_max = 50.0
    vy_min = -100.0
    vy_max = 50.0
    omega_min = -0.5
    omega_max = 0.5

    theta_max_loose = np.pi / 4
    LANDING_NODES = 5
    TIGHT_THETA_DEG = 2.0
    tight_theta = np.deg2rad(TIGHT_THETA_DEG)
    TIGHT_ALPHA_DEG = 2.0
    tight_alpha = np.deg2rad(TIGHT_ALPHA_DEG)

    initial_tf_guess = 15.0

    epsilon = 20.0
    w_time = 10.0
    w_thrust = 1e-10
    w_gimbal = 0.05
    w_gimbal_rate = 0.25
    w_theta = 2.0
    w_alt_thrust = 6e-6
    w_landing = 4000.0
    w_v = 50.0

    clearance_zone_factor = 2.0
    clearance_zone = rocket_height * clearance_zone_factor

    nozzle_ground_threshold = -0.1
    hover_thrust_safety_factor = 0.6
    dv_available_margin = 0.6
    initial_theta_max_deg = 25
    initial_omega_max_deg = 35
    initial_x_max = 800
    dv_gravity_loss_factor = 1.5
    dv_time_margin = 5.0

    landing_pos_tol = 0.10
    landing_vel_tol = 0.15
    min_nozzle_tol = -0.30
    skidding_x_threshold = 5.0

    # grid size for testing
    x_grid_min = -10.0
    x_grid_max = 10.0
    x_grid_step = 5.0
    h_grid_min = 0.0
    h_grid_max = 100.0
    h_grid_step = 25.0


    # initial conditions (constant for all tests)
    test_vx_0 = 0.0
    test_vy_0 = 0.0
    test_theta_0 = 0.0
    test_omega_0 = 0.0


config = RocketConfig()


def calculate_com_and_I(curr_mass):
    fuel_remaining = max(curr_mass - config.m_dry, 0.0)
    com_from_nozzle = (config.m_dry * config.dry_cm_from_nozzle +
                       fuel_remaining * config.fuel_cm_from_nozzle) / curr_mass

    I_dry_cm = (1 / 12) * config.m_dry * config.rocket_height ** 2
    dry_distance = config.dry_cm_from_nozzle - com_from_nozzle
    I_dry = I_dry_cm + config.m_dry * dry_distance ** 2

    I_fuel_cm = (1 / 12) * fuel_remaining * config.fuel_tank_height ** 2 if fuel_remaining > 0 else 0
    fuel_distance = config.fuel_cm_from_nozzle - com_from_nozzle
    I_fuel = I_fuel_cm + fuel_remaining * fuel_distance ** 2

    I_z = I_dry + I_fuel
    return com_from_nozzle, I_z


def approx_controllable(initial_state, config):
    x, y, vx, vy, theta, omega, m = initial_state
    fuel_mass = m - config.m_dry

    if fuel_mass < 1e-6:
        print("No usable fuel --> cannot control attitude, position or rate.")
        return False

    com, _ = calculate_com_and_I(m)
    nozzle_y = y - com
    if nozzle_y < config.nozzle_ground_threshold:
        return False

    hover_thrust_needed = m * config.g
    if config.T_max < config.hover_thrust_safety_factor * hover_thrust_needed:
        print("Thrust too low even to hover.")
        return False

    dv_available = config.v_e * np.log(m / config.m_dry)
    speed = np.sqrt(vx ** 2 + vy ** 2)
    dv_required_rough = speed + config.dv_gravity_loss_factor * config.g * (
            abs(vy) / config.g + config.dv_time_margin)

    if dv_available < config.dv_available_margin * dv_required_rough:
        print(f"Delta-v budget too low: {dv_available:.1f} <~ {dv_required_rough:.1f} m/s")
        return False

    if (abs(theta) > np.deg2rad(config.initial_theta_max_deg) or
            abs(omega) > np.deg2rad(config.initial_omega_max_deg)):
        print("Initial attitude or rate too extreme for recovery with limited control.")
        return False

    if abs(x) > config.initial_x_max:
        print("Too far downrange for realistic recovery.")
        return False

    return True


def solve_optimal_landing(initial_state):
    print("Setting up CasADi + IPOPT NLP...")
    opti = ca.Opti()
    N = config.N

    X = opti.variable(7, N + 1)
    U = opti.variable(2, N + 1)
    tf = opti.variable()
    dt = tf / N

    xs, ys, vxs, vys, thetas, omegas, ms = X[0, :], X[1, :], X[2, :], X[3, :], X[4, :], X[5, :], X[6, :]
    Ts, alphas = U[0, :], U[1, :]

    def calc_com_I_casadi(m):
        fuel_remaining = m - config.m_dry
        com_from_nozzle = (config.m_dry * config.dry_cm_from_nozzle +
                           fuel_remaining * config.fuel_cm_from_nozzle) / m
        I_dry_cm = (1 / 12) * config.m_dry * config.rocket_height ** 2
        dry_dist = config.dry_cm_from_nozzle - com_from_nozzle
        I_dry = I_dry_cm + config.m_dry * dry_dist ** 2
        I_fuel_cm = (1 / 12) * fuel_remaining * config.fuel_tank_height ** 2
        fuel_dist = config.fuel_cm_from_nozzle - com_from_nozzle
        I_fuel = I_fuel_cm + fuel_remaining * fuel_dist ** 2
        I_z = I_dry + I_fuel
        return com_from_nozzle, I_z

    def rocket_dynamics_casadi(state, control):
        x, y, vx, vy, theta, omega, m = state[0], state[1], state[2], state[3], state[4], state[5], state[6]
        T, alpha = control[0], control[1]
        l_t, I_z = calc_com_I_casadi(m)
        thrust_angle = theta + alpha
        dx = vx
        dy = vy
        dvx = (T / m) * ca.sin(thrust_angle)
        dvy = (T / m) * ca.cos(thrust_angle) - config.g
        dtheta = omega
        domega = -(l_t * T / I_z) * ca.sin(alpha)
        dm = -T / config.v_e
        return ca.vertcat(dx, dy, dvx, dvy, dtheta, domega, dm)

    for k in range(N):
        s_k = X[:, k]
        s_kp1 = X[:, k + 1]
        u_k = U[:, k]
        u_kp1 = U[:, k + 1]
        f_k = rocket_dynamics_casadi(s_k, u_k)
        f_kp1 = rocket_dynamics_casadi(s_kp1, u_kp1)
        opti.subject_to(s_kp1 == s_k + 0.5 * dt * (f_k + f_kp1))

    opti.subject_to(X[:, 0] == initial_state)
    opti.subject_to(X[0, -1] == 0)
    opti.subject_to(X[2, -1] == 0)
    opti.subject_to(X[3, -1] == 0)
    opti.subject_to(X[4, -1] == 0)
    opti.subject_to(X[5, -1] == 0)

    com_term, _ = calc_com_I_casadi(X[6, -1])
    opti.subject_to(X[1, -1] == com_term)
    opti.subject_to(U[1, -1] == 0.0)

    opti.subject_to(opti.bounded(config.tf_min, tf, config.tf_max))
    opti.subject_to(opti.bounded(0.0, ys, ca.inf))
    opti.subject_to(opti.bounded(config.vx_min, vxs, config.vx_max))
    opti.subject_to(opti.bounded(config.vy_min, vys, config.vy_max))
    opti.subject_to(opti.bounded(config.omega_min, omegas, config.omega_max))
    opti.subject_to(opti.bounded(config.m_dry, ms, initial_state[6]))
    opti.subject_to(opti.bounded(config.T_min, Ts, config.T_max))

    landing_nodes = config.LANDING_NODES
    tight_theta = config.tight_theta
    tight_alpha = config.tight_alpha
    for k in range(N + 1):
        if k >= N + 1 - landing_nodes:
            opti.subject_to(opti.bounded(-tight_theta, thetas[k], tight_theta))
            opti.subject_to(opti.bounded(-tight_alpha, alphas[k], tight_alpha))
        else:
            opti.subject_to(opti.bounded(-config.theta_max_loose, thetas[k], config.theta_max_loose))
            opti.subject_to(opti.bounded(-config.alpha_max, alphas[k], config.alpha_max))

    J_time = config.w_time * tf
    J_thrust = config.w_thrust * dt * ca.sumsqr(Ts)
    J_gimbal = config.w_gimbal * dt * ca.sumsqr(alphas)
    gimbal_rates = ca.diff(alphas) / dt
    J_gimbal_rate = config.w_gimbal_rate * dt * ca.sumsqr(gimbal_rates)
    J_theta = config.w_theta * dt * ca.sumsqr(thetas)

    com_heights = [calc_com_I_casadi(ms[k])[0] for k in range(N + 1)]
    com_heights = ca.horzcat(*com_heights)
    nozzle_heights = ys - com_heights
    clearance_zone = config.clearance_zone

    ground_proximity = ca.fmax(0, 1.0 - nozzle_heights / clearance_zone) ** 3
    ground_violation = ca.fmax(0, -nozzle_heights)
    h_nozzle_safe = ca.fmax(config.epsilon, nozzle_heights)

    J_inverse_h = config.w_v * dt * ca.sum2((vxs ** 2 + vys ** 2) / h_nozzle_safe)
    J_ground = 1e8 * ca.sumsqr(ground_violation)
    J_landing_theta = config.w_landing * dt * ca.sumsqr(thetas * ground_proximity)
    J_landing_gimbal = config.w_landing * dt * ca.sumsqr(alphas * ground_proximity)

    y_norm = ys / initial_state[1]
    J_alt_thrust = config.w_alt_thrust * dt * ca.sum2((Ts ** 2) * y_norm)

    J = (J_time + J_thrust + J_gimbal + J_gimbal_rate + J_theta +
         J_alt_thrust + J_ground + J_landing_theta + J_landing_gimbal + J_inverse_h)
    opti.minimize(J)

    opti.set_initial(tf, config.initial_tf_guess)

    target_com, _ = calculate_com_and_I(config.m_dry)
    target_state = np.array([0, target_com, 0, 0, 0, 0, config.m_dry])
    for i in range(7):
        guess_traj = np.linspace(initial_state[i], target_state[i], N + 1)
        opti.set_initial(X[i, :], guess_traj)

    hover_thrust = initial_state[6] * config.g
    opti.set_initial(Ts, np.clip(hover_thrust, config.T_min, config.T_max))
    opti.set_initial(alphas, 0.0)

    p_opts = {"expand": True}
    s_opts = {"max_iter": 500, "tol": 1e-6, "print_level": 5}
    opti.solver("ipopt", p_opts, s_opts)

    print("\nStarting Optimization with IPOPT...")
    try:
        sol = opti.solve()
        success = True
        print("\n--- Solution Report ---")
        print(f"  Converged:        {success}")
    except RuntimeError:
        sol = opti.debug
        success = False
        print("\n  WARNING: Optimizer did not fully converge.")

    states_sol = sol.value(X).T
    controls_sol = sol.value(U).T
    tf_sol = sol.value(tf)

    com_term, _ = calculate_com_and_I(states_sol[-1, 6])
    final_pos_err = np.linalg.norm([states_sol[-1, 0] - 0, states_sol[-1, 1] - com_term])
    final_vel_err = np.linalg.norm(states_sol[-1, 2:4])

    print(f"  Landing time:     {tf_sol:.3f} s")
    print(f"  Final pos error:  {final_pos_err:.4f} m")
    print(f"  Final vel error:  {final_vel_err:.4f} m/s")
    print(f"  Fuel consumed:    {initial_state[6] - states_sol[-1, 6]:.2f} kg")
    return states_sol, controls_sol, tf_sol, success


# ==========================================
# Main Execution - Grid Test with Progress Counter
# ==========================================
if __name__ == "__main__":
    print("--- Time-Optimal Rocket Landing Solver (Grid Test) ---")

    vx_0 = config.test_vx_0
    vy_0 = config.test_vy_0

    initial_m = config.m_dry + config.m_fuel
    initial_com, _ = calculate_com_and_I(initial_m)

    print(f"\nFixed initial conditions for all grid points:")
    print(f"  Vx: {vx_0} m/s")
    print(f"  Vy: {vy_0} m/s")
    print(f"  Theta: {config.test_theta_0} rad")
    print(f"  Omega: {config.test_omega_0} rad/s")
    print(f"  Fuel mass   : {config.m_fuel:6.1f} kg")
    print(f"  Dry mass    : {config.m_dry:6.1f} kg")
    print(f"  Initial CoM : {initial_com:6.2f} m from nozzle")

    x_grid = np.arange(config.x_grid_min, config.x_grid_max + config.x_grid_step / 2, config.x_grid_step)
    h_grid = np.arange(config.h_grid_min, config.h_grid_max + config.h_grid_step / 2, config.h_grid_step)

    total_simulations = len(x_grid) * len(h_grid)
    current = 0

    results = []

    print(f"\nTotal simulations to run: {total_simulations}\n")

    for x_0_orig in x_grid:
        for h_0 in h_grid:
            current += 1
            progress_pct = (current / total_simulations) * 100
            progress_str = f"Progress: [{current}/{total_simulations}] {progress_pct:5.1f}%"

            if IN_NOTEBOOK:
                clear_output(wait=True)
                print(progress_str)
            else:
                sys.stdout.write(f"\r{progress_str}")
                sys.stdout.flush()
            print(f"\nTesting initial position: x={x_0_orig:.1f}, nozzle_alt={h_0:.1f} m")
            print("") # empty line for better readability

            x_0 = 0.0001 if abs(x_0_orig) < 1e-6 else x_0_orig
            vx_0_pert = 0.0001 if abs(vx_0) < 1e-6 else vx_0
            vy_0_pert = 0.0001 if abs(vy_0) < 1e-6 else vy_0

            s0 = np.array([
                x_0,
                h_0 + initial_com,
                vx_0_pert,
                vy_0_pert,
                config.test_theta_0,
                config.test_omega_0,
                initial_m
            ])

            print("\nChecking approximate controllability...")
            if not approx_controllable(s0, config):
                print("--> Not controllable.")
                results.append((x_0_orig, h_0, False))
                continue

            print("--> Approximate check passed. Starting optimization...")
            states, controls, tf, success = solve_optimal_landing(s0)

            if not success:
                print("\nOptimization did NOT converge.")
                results.append((x_0_orig, h_0, False))
                continue

            print("\nOptimization finished. Evaluating solution quality...")

            coms = np.array([calculate_com_and_I(m)[0] for m in states[:, 6]])
            y_nozzle = states[:, 1] - coms
            x_coms = states[:, 0]

            com_term, _ = calculate_com_and_I(states[-1, 6])
            error_x = states[-1, 0] - 0.0
            error_y = states[-1, 1] - com_term

            final_pos_err = np.linalg.norm([error_x, error_y])
            final_vel_err = np.linalg.norm(states[-1, 2:4])
            min_nozzle = np.min(y_nozzle)

            skidding_detected = False
            for i in range(len(y_nozzle) - 1):
                if y_nozzle[i] < 0.1 and abs(x_coms[i]) > config.skidding_x_threshold:
                    skidding_detected = True
                    break

            LANDING_GOOD = (
                final_pos_err < config.landing_pos_tol and
                final_vel_err < config.landing_vel_tol and
                min_nozzle >= config.min_nozzle_tol and
                not skidding_detected
            )

            if LANDING_GOOD:
                print("\nLanding looks good within tolerances!")
                print(f"  X-offset:      {error_x:6.3f} m")
                print(f"  Y-offset:      {error_y:6.3f} m")
                print(f"  Velocity err:  {final_vel_err:6.3f} m/s")
                print(f"  Min nozzle:    {min_nozzle:6.3f} m")
                results.append((x_0_orig, h_0, True))
            else:
                print("\nSolution found, but landing quality NOT acceptable:")
                issues = []
                if abs(error_x) > config.landing_pos_tol: issues.append(f"X error: {error_x:.3f} m")
                if abs(error_y) > config.landing_pos_tol: issues.append(f"Y error: {error_y:.3f} m")
                if final_pos_err >= config.landing_pos_tol: issues.append(f"pos err = {final_pos_err:.3f} m")
                if final_vel_err >= config.landing_vel_tol: issues.append(f"vel err = {final_vel_err:.3f} m/s")
                if min_nozzle < config.min_nozzle_tol: issues.append(f"ground clip = {abs(min_nozzle):.3f} m")
                if skidding_detected: issues.append("skidding detected")
                for issue in issues:
                    print(f"  • {issue}")
                results.append((x_0_orig, h_0, False))

    print("\n\nAll simulations completed.")
    print("Generating success grid plot...")

    # Plot
    fig, ax = plt.subplots(figsize=(12, 8))

    success_points = [(x, h) for x, h, succ in results if succ]
    failure_points = [(x, h) for x, h, succ in results if not succ]

    if success_points:
        sx, sh = zip(*success_points)
        ax.plot(sx, sh, 'go', markersize=8, label='Success')

    if failure_points:
        fx, fh = zip(*failure_points)
        ax.plot(fx, fh, 'rx', markersize=8, label='Failure')

    ax.set_xlabel('Initial x (m)')
    ax.set_ylabel('Initial Nozzle Altitude (m)')
    ax.set_title('Landing Success Grid\n(fixed velocity & attitude conditions)')
    ax.grid(True)
    ax.legend(loc='upper left', fontsize=10)

    # info box next to plot
    textstr = (
        "Fixed initial conditions (same for all points):\n\n"
        rf"  Horizontal velocity   $Vx_0$"+f" = {vx_0:>+6.1f} m/s\n"
        fr"  Vertical velocity     $Vy_0$"+f" = {vy_0:>+6.1f} m/s\n"
        fr"  Pitch angle            $\theta_0$"+f" = {config.test_theta_0:>+6.1f} rad\n"
        fr"  Pitch rate             $\omega_0$"+f" = {config.test_omega_0:>+6.1f} rad/s\n"
        f"  Dry mass                  = {config.m_dry:>6.0f} kg\n"
        f"  Fuel mass                 = {config.m_fuel:>6.0f} kg\n"
        f"  Total initial mass        = {initial_m:>6.0f} kg\n"
        f"  Initial CoM from nozzle   =  {initial_com:>5.2f} m\n"
        "\n"
        rf"Grid: $x_0$ $\in$ [{config.x_grid_min}, {config.x_grid_max}] m  (step {config.x_grid_step} m)"+"\n"
        rf"      $h_0$ $\in$ [{config.h_grid_min}, {config.h_grid_max}]  m  (step {config.h_grid_step} m) (nozzle)"
    )

    props = dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='gray')
    ax.text(1.05, 0.98, textstr,
            transform=ax.transAxes,
            fontsize=10,
            family='monospace',
            verticalalignment='top',
            horizontalalignment='left',
            bbox=props)

    plt.subplots_adjust(right=0.72)
    plt.savefig(f"../results/convergence_test.png", dpi=150, bbox_inches='tight')
    plt.show()
    